In [1]:
# ------------------ CLEAN ENV SETUP ------------------

!pip uninstall -y torch torchvision torchaudio transformers sentence-transformers numpy faiss-cpu torchcodec -q

# Core numerical + ML stack (stable versions)
!pip install numpy==1.26.4 -q
!pip install torch==2.2.2 --index-url https://download.pytorch.org/whl/cpu -q
!pip install transformers==4.40.2 -q
!pip install sentence-transformers==2.7.0 -q

# Vector DB
!pip install faiss-cpu==1.7.4 -q

# PDF processing (used in your RAGEngine)
!pip install pypdf -q

# Groq client (LLM backend)
!pip install groq -q
!pip install openai -q

#There will be some warnings and Errors - It's ok Just ignore and go ahead


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 86.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
accelerate 1.14.0 requires torch>=2.0.0, which is not installed.
fastai 2.8.7 requires torch<3,>=1.10, which is not installed.
fastai 2.8.7 requires torchvision>=0.11, which is not installed.
peft 0.19.1 requires torch>=1.13.0, which is not installed.
peft 0.19.1 requires transformers, which is not installed.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 

In [2]:
import sys
!{sys.executable} -m pip install faiss-cpu==1.8.0
!{sys.executable} -m pip install pdfplumber -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.0/27.0 MB 9.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 22.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastai 2.8.7 requires torchvision>=0.11, which is not installed.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.


In [ ]:
import os
os.kill(os.getpid(), 9)

In [49]:
#Now Do not run any command which is upto this cell
#Go ahead with importing 3 documents
#1. Guidelines_pdf.pdf
#2. int_rate_new.xlsx
#3. benchmark_demo.csv
#This is colab notebook so add it in the sample_data folder of colab environment

In [50]:
import faiss
import numpy as np
from groq import Groq
from openai import OpenAI
import pandas as pd
import datetime
from datetime import datetime
import json
import re
from sentence_transformers import SentenceTransformer

In [51]:
import os
os.environ["TRANSFORMERS_NO_TORCHCODEC"] = "1"

In [52]:
import os
os.environ["USE_TORCH"] = "1"

In [53]:
TODAY_STR = datetime.today().strftime("%d %b %Y")   # e.g. "07 Mar 2026"
TODAY_ISO = datetime.today().strftime("%Y-%m-%d")   # e.g. "2026-03-07"

In [54]:
#False the respective ablation case to run the ablation study for that particular case
ABLATION_TENURE_ENGINE   = True   # Ablation 1
ABLATION_ROLLING_YEAR    = True   # Ablation 2
ABLATION_EXACT_N         = True   # Ablation 3
ABLATION_RATE_CARD       = True   # Ablation 4
ABLATION_PREMATURE_OVERRIDE = False  # Ablation 5

In [55]:
llm_used="meta-llama/llama-4-scout-17b-16e-instruct"#"llama-3.1-8b-instant","qwen/qwen3-32b","meta-llama/llama-4-scout-17b-16e-instruct"#"openai/gpt-oss-20b"# "llama-3.3-70b-versatile" # "openai/gpt-oss-20b" # "openai/gpt-oss-120b"
MODEL_CONFIG = {
    "router": llm_used,#"openai/gpt-oss-120b",#"llama-3.3-70b-versatile",
    "agent1": llm_used,#"openai/gpt-oss-120b",
    "agent2": llm_used,#"openai/gpt-oss-120b",
    "rag_rules": llm_used,#"openai/gpt-oss-120b",
    "agent_explainer": llm_used,#"openai/gpt-oss-120b",
    "agent_rate_query": llm_used,#"openai/gpt-oss-120b",
    "default": llm_used#"openai/gpt-oss-120b"
}

def get_model(agent_name):
    return MODEL_CONFIG.get(agent_name, MODEL_CONFIG["default"])

In [56]:
def extract_json_from_text(text):
    """
    Extracts JSON from LLM response.
    Handles: preamble text, code fences, unescaped newlines, TRUNCATED JSON.
    """
    if not text or not text.strip():
        raise ValueError("❌ Empty response from LLM")

    # Strip code fences
    text = re.sub(r"```[\s\S]*?```", lambda m: m.group(0).strip("`").strip(), text)

     # ── NEW: if text starts mid-JSON without opening {, prepend it ──
    stripped = text.strip()
    if stripped and stripped[0] == '"':
        text = "{" + stripped
        # also close it if not closed
        if not stripped.rstrip().endswith("}"):
            text = text + "}"

    # Try direct parse (clean compact JSON)
    try:
        return json.loads(text.strip())
    except Exception:
        pass

    # Find complete JSON object (handles preamble/postamble)
    json_match = re.search(r"\{[\s\S]*\}", text)
    if json_match:
        json_str = json_match.group(0)
    else:
        # No closing brace → truncated; grab from first { to end
        json_match = re.search(r"\{[\s\S]*", text)
        if not json_match:
            raise ValueError(f"❌ No JSON object found in text:\n{text}")
        json_str = json_match.group(0)

    # Fix unescaped newlines inside string values
    def fix_newlines(s):
        inside = False
        result = []
        for i, ch in enumerate(s):
            if ch == '"' and (i == 0 or s[i-1] != '\\'):
                inside = not inside
            result.append('\\n' if inside and ch == '\n' else ch)
        return ''.join(result)

    json_str = fix_newlines(json_str)

    try:
        return json.loads(json_str)
    except json.JSONDecodeError as e:
        # ── TRUNCATION RECOVERY ─────────────────────────────────
        print("⚠️ JSON truncated — attempting field recovery...")
        recovered = {}
        patterns = [
            (r'"principal"\s*:\s*([0-9]+(?:\.[0-9]+)?)',     "principal",          float),
            (r'"start_date"\s*:\s*"([^"]+)"',                "start_date",         str),
            (r'"tenure_value"\s*:\s*([0-9]+(?:\.[0-9]+)?)',  "tenure_value",       float),
            (r'"tenure_unit"\s*:\s*"([^"]+)"',               "tenure_unit",        str),
            (r'"withdrawal_date"\s*:\s*"([^"]+)"',           "withdrawal_date",    str),
            (r'"withdrawal_date"\s*:\s*(null)',               "withdrawal_date",    lambda x: None),
            (r'"holding_days"\s*:\s*(null|\d+)',             "holding_days",       lambda x: None if x=="null" else int(x)),
            (r'"payout_type"\s*:\s*"([^"]+)"',               "payout_type",        str),
            (r'"is_premature"\s*:\s*(true|false)',            "is_premature",       lambda x: x=="true"),
            (r'"citizenship_status"\s*:\s*"([^"]+)"',        "citizenship_status", str),
            (r'"nri_status"\s*:\s*"([^"]+)"',                "nri_status",         str),
            (r'"deposit_amount"\s*:\s*"([^"]+)"',            "deposit_amount",     str),
        ]
        for pattern, key, cast in patterns:
            m = re.search(pattern, json_str)
            if m:
                try:
                    recovered[key] = cast(m.group(1))
                except Exception:
                    pass

        if recovered.get("principal") and recovered.get("start_date") and recovered.get("tenure_value"):
            print(f"✅ Recovery succeeded: {list(recovered.keys())}")
            recovered.setdefault("tenure_unit",        "years")
            recovered.setdefault("withdrawal_date",    None)
            recovered.setdefault("holding_days",       None)
            recovered.setdefault("payout_type",        "cumulative")
            recovered.setdefault("is_premature",       False)
            recovered.setdefault("citizenship_status", "regular")
            recovered.setdefault("nri_status",         "Non NRI")
            recovered.setdefault("deposit_amount",     "<3CR")
            return recovered

        raise ValueError(
            f"❌ JSON parsing failed and field recovery insufficient.\n"
            f"Recovered: {list(recovered.keys())}\n"
            f"Raw (first 300): {json_str[:300]}\nError: {e}"
        )

In [57]:
def sanitize_query_for_llm(user_query):
    s = user_query
    s = s.replace("₹", "Rs.")
    s = s.replace("\u20b9", "Rs.")
    s = s.replace("broke my",  "closed my")
    s = s.replace("break my",  "close my")
    s = s.replace("broken my", "closed my")
    s = s.replace("I broke",   "I closed")
    s = s.replace("broke the", "closed the")
    return s

In [58]:
# =========================================================
# 🔹 PYTHON TENURE ENGINE
# All tenure math is done here — zero LLM involvement.
# =========================================================
import calendar
from dateutil.relativedelta import relativedelta
from datetime import datetime, timedelta


def compute_maturity_date(start_date_str, tenure_value, tenure_unit):
    """
    Compute exact maturity date and tenure_days purely in Python.

    Args:
        start_date_str : "YYYY-MM-DD"
        tenure_value   : numeric (int or float). For fractional years pass as float.
        tenure_unit    : "days" | "months" | "years" | "quarters"

    Rules:
        days     → start + N days (exact timedelta)
        months   → relativedelta(months=N)  — EOM-safe
        years    → relativedelta(years=N)   — EOM-safe (29 Feb → 28 Feb non-leap)
        quarters → convert to months (N*3), then same as months
        fractional years (e.g. 1.5) → convert to total months (round(N*12))

    Examples:
        30 Nov 2023 + 9 months  = 30 Aug 2024  (274 days)  ✅
        29 Feb 2024 + 2 years   = 28 Feb 2026  (730 days)  ✅
        31 Jan 2024 + 1 year    = 31 Jan 2025  (366 days)  ✅
        01 Jan 2024 + 4 quarters= 01 Jan 2025  (366 days)  ✅
        01 Jan 2024 + 730 days  = 31 Dec 2025  (730 days)  ✅

    Returns:
        (maturity_date_str: str "YYYY-MM-DD", tenure_days: int)
    """
    start = datetime.strptime(start_date_str, "%Y-%m-%d").date()
    value = float(tenure_value)
    unit  = str(tenure_unit).lower().strip()

    def eom_snap(d):
        last_day = calendar.monthrange(d.year, d.month)[1]
        return d.replace(day=min(d.day, last_day))

    if unit == "days":
        maturity = start + timedelta(days=int(value))

    elif unit == "months":
        raw = (datetime.strptime(start_date_str, "%Y-%m-%d")
               + relativedelta(months=int(value))).date()
        maturity = eom_snap(raw)

    elif unit == "years":
        if value != int(value):
            # Fractional years → convert to total months
            total_months = int(round(value * 12))
            raw = (datetime.strptime(start_date_str, "%Y-%m-%d")
                   + relativedelta(months=total_months)).date()
        else:
            raw = (datetime.strptime(start_date_str, "%Y-%m-%d")
                   + relativedelta(years=int(value))).date()
        maturity = eom_snap(raw)

    elif unit == "quarters":
        total_months = int(value) * 3
        raw = (datetime.strptime(start_date_str, "%Y-%m-%d")
               + relativedelta(months=total_months)).date()
        maturity = eom_snap(raw)

    else:
        raise ValueError(f"Unknown tenure_unit: '{unit}'. Use days/months/years/quarters.")

    tenure_days = (maturity - start).days
    return maturity.strftime("%Y-%m-%d"), tenure_days


def parse_tenure_from_llm(extracted):
    """
    Given Agent-1 JSON, compute authoritative tenure_days and maturity_date using Python.

    LLM is only trusted for:
        tenure_value (the number)
        tenure_unit  (days / months / years / quarters)

    Everything else (tenure_days, tenure_years, maturity_date) is computed here.

    Falls back to tenure_years if tenure_unit not present (legacy support).
    """
    start_date_str = extracted.get("start_date")
    if not start_date_str:
        return extracted  # nothing to do

    tenure_value = extracted.get("tenure_value")
    tenure_unit  = extracted.get("tenure_unit")

    # ── Primary path: LLM sent tenure_value + tenure_unit ─────────────────────
    if tenure_value is not None and tenure_unit:
        try:
            maturity_str, tenure_days = compute_maturity_date(
                start_date_str, tenure_value, tenure_unit
            )
            print(f"Tenure engine: {tenure_value} {tenure_unit} from {start_date_str} "
                  f"→ maturity {maturity_str} ({tenure_days} days)")
            extracted["tenure_days"]  = tenure_days
            extracted["tenure_years"] = round(tenure_days / 365, 4)
            extracted["maturity_date"] = maturity_str
            return extracted
        except Exception as e:
            print(f"Tenure engine error (primary): {e}")

    # ── Fallback: LLM sent tenure_months (legacy) ─────────────────────────────
    tenure_months = extracted.get("tenure_months")
    if tenure_months is not None:
        try:
            maturity_str, tenure_days = compute_maturity_date(
                start_date_str, tenure_months, "months"
            )
            print(f"Tenure engine (months fallback): {tenure_months}m from {start_date_str} "
                  f"→ maturity {maturity_str} ({tenure_days} days)")
            extracted["tenure_days"]  = tenure_days
            extracted["tenure_years"] = round(tenure_days / 365, 4)
            extracted["maturity_date"] = maturity_str
            return extracted
        except Exception as e:
            print(f"Tenure engine error (months fallback): {e}")

    # ── Last fallback: LLM sent tenure_years only (legacy) ────────────────────
    tenure_years = extracted.get("tenure_years")
    if tenure_years is not None:
        try:
            maturity_str, tenure_days = compute_maturity_date(
                start_date_str, float(tenure_years), "years"
            )
            print(f"Tenure engine (years fallback): {tenure_years}y from {start_date_str} "
                  f"→ maturity {maturity_str} ({tenure_days} days)")
            extracted["tenure_days"]  = tenure_days
            extracted["tenure_years"] = round(tenure_days / 365, 4)
            extracted["maturity_date"] = maturity_str
            return extracted
        except Exception as e:
            print(f"Tenure engine error (years fallback): {e}")

    return extracted


In [59]:
# =========================================================
# TENURE ENGINE — pure Python, zero LLM math
# =========================================================
import calendar
from dateutil.relativedelta import relativedelta
from datetime import datetime, timedelta


def compute_maturity_date(start_date_str, tenure_value, tenure_unit):
    start = datetime.strptime(start_date_str, "%Y-%m-%d").date()
    value = float(tenure_value)
    unit  = str(tenure_unit).lower().strip()

    def eom_snap(d):
        last_day = calendar.monthrange(d.year, d.month)[1]
        return d.replace(day=min(d.day, last_day))

    if unit == "days":
        maturity = start + timedelta(days=int(value))

    elif unit == "months":
        raw      = (datetime.strptime(start_date_str, "%Y-%m-%d") + relativedelta(months=int(value))).date()
        maturity = eom_snap(raw)

    elif unit == "years":
        if value != int(value):
            total_months = int(round(value * 12))
            raw = (datetime.strptime(start_date_str, "%Y-%m-%d") + relativedelta(months=total_months)).date()
        else:
            raw = (datetime.strptime(start_date_str, "%Y-%m-%d") + relativedelta(years=int(value))).date()
        maturity = eom_snap(raw)

    elif unit == "quarters":
        raw      = (datetime.strptime(start_date_str, "%Y-%m-%d") + relativedelta(months=int(value) * 3)).date()
        maturity = eom_snap(raw)

    else:
        raise ValueError(f"Unknown tenure_unit: '{unit}'. Use days/months/years/quarters.")

    return maturity.strftime("%Y-%m-%d"), (maturity - start).days


def parse_tenure_from_llm(extracted):
    """Compute tenure_days + maturity_date in Python. LLM only provides tenure_value + tenure_unit."""
    start_date_str = extracted.get("start_date")
    if not start_date_str:
        return extracted

    tv = extracted.get("tenure_value")
    tu = extracted.get("tenure_unit")

    # Primary: tenure_value + tenure_unit (new schema)
    if tv is not None and tu:
        try:
            maturity_str, tenure_days = compute_maturity_date(start_date_str, tv, tu)
            print(f"Tenure engine: {tv} {tu} from {start_date_str} → maturity {maturity_str} ({tenure_days} days)")
            extracted["tenure_days"]   = tenure_days
            extracted["tenure_years"]  = round(tenure_days / 365, 4)
            extracted["maturity_date"] = maturity_str
            return extracted
        except Exception as e:
            print(f"Tenure engine error (primary): {e}")

    # Fallback: tenure_months (legacy)
    tm = extracted.get("tenure_months")
    if tm is not None:
        try:
            maturity_str, tenure_days = compute_maturity_date(start_date_str, tm, "months")
            extracted["tenure_days"]   = tenure_days
            extracted["tenure_years"]  = round(tenure_days / 365, 4)
            extracted["maturity_date"] = maturity_str
            return extracted
        except Exception as e:
            print(f"Tenure engine error (months fallback): {e}")

    # Fallback: tenure_years (legacy)
    ty = extracted.get("tenure_years")
    if ty is not None:
        try:
            maturity_str, tenure_days = compute_maturity_date(start_date_str, float(ty), "years")
            extracted["tenure_days"]   = tenure_days
            extracted["tenure_years"]  = round(tenure_days / 365, 4)
            extracted["maturity_date"] = maturity_str
            return extracted
        except Exception as e:
            print(f"Tenure engine error (years fallback): {e}")

    return extracted



In [60]:
def run_agent1(user_query):
    try:
        safe_query = sanitize_query_for_llm(user_query)

        # ── TODAY injection gated by Ablation 6 ───────────────────────────────
        today_line = (
            f"Today's date is {TODAY_STR} ({TODAY_ISO} in YYYY-MM-DD format).\n"
            f"Use this if the user does not specify a booking date or says "
            f'"today", "now", "breaking today", "closing today".'
        )

        full_prompt = f"""{today_line}

{agent1_prompt}

USER QUERY:
{safe_query}

Return ONLY the JSON. No explanation."""

        # ── PRIMARY LLM CALL ──────────────────────────────────────────────────
        resp = client.chat.completions.create(
            model=get_model("agent1"),
            messages=[
                {"role": "system", "content": "You are a precise FD parameter extraction agent. Return ONLY valid JSON. No explanation, no text before or after the JSON."},
                {"role": "user",   "content": full_prompt}
            ],
            temperature=0.1,
            max_tokens=500
        ).choices[0].message.content

        print(f"PRIMARY RESP:\n{resp[:500] if resp else 'EMPTY RESPONSE'}")
        print("=" * 50)

        extracted = extract_json_from_text(resp)

        # ── RETRY if primary failed ────────────────────────────────────────────
        if not extracted:
            print("⚠️ Agent-1 primary call returned no JSON. Retrying...")

            retry_today = (
                f"Today's date is {TODAY_STR} ({TODAY_ISO}).\n"
            )

            #retry_prompt = f"""{retry_today}Extract FD details. Return ONLY a JSON object, nothing else.
            retry_today = f"Today's date is {TODAY_STR} ({TODAY_ISO}).\n"
            retry_prompt = f"""{retry_today}
Query: {safe_query}

{{
  "principal": <number>,
  "start_date": "<YYYY-MM-DD>",
  "tenure_value": <number>,
  "tenure_unit": "<days|months|years|quarters>",
  "withdrawal_date": "<YYYY-MM-DD or null>",
  "holding_days": null,
  "payout_type": "<cumulative|quarterly_payout|monthly_payout|half_yearly_payout>",
  "is_premature": <true|false>,
  "citizenship_status": "<regular|senior>",
  "nri_status": "<NRI|Non NRI>",
  "deposit_amount": "<<3CR|>=3CR>"
}}"""

            retry_resp = client.chat.completions.create(
                model=get_model("agent1"),
                messages=[
                    {"role": "system", "content": "Return ONLY valid JSON. No other text."},
                    {"role": "user",   "content": retry_prompt}
                ],
                temperature=0.0,
                max_tokens=400
            ).choices[0].message.content

            print(f"RETRY RESP:\n{retry_resp[:500] if retry_resp else 'EMPTY RESPONSE'}")
            print("=" * 50)
            extracted = extract_json_from_text(retry_resp)

        if not extracted:
            return {"route": "agent1_error", "error_type": "json_parse_failed",
                    "final_answer": "I couldn't parse your FD details. Please try rephrasing."}

        # ── TENURE ENGINE: 100% Python (Ablation 1) ───────────────────────────
        if ABLATION_TENURE_ENGINE:
            # Full Python tenure engine: computes exact calendar-aware
            # tenure_days and maturity_date from tenure_value + tenure_unit
            extracted = parse_tenure_from_llm(extracted)
        else:
            # Ablation 1 OFF: use LLM's raw tenure_value + tenure_unit
            # but compute a naive tenure_days so downstream doesn't break
            if extracted.get("tenure_days") is None:
                tv = extracted.get("tenure_value")
                tu = str(extracted.get("tenure_unit", "years")).lower().strip()
                if tv is not None:
                    tv = float(tv)
                    if tu == "years":
                        extracted["tenure_days"] = int(round(tv * 365))
                    elif tu == "months":
                        extracted["tenure_days"] = int(round(tv * 30.44))
                    elif tu == "quarters":
                        extracted["tenure_days"] = int(round(tv * 91.31))
                    elif tu == "days":
                        extracted["tenure_days"] = int(tv)
            # Ensure int, not string
            if extracted.get("tenure_days") is not None:
                try:
                    extracted["tenure_days"] = int(float(extracted["tenure_days"]))
                except:
                    extracted["tenure_days"] = None
            # tenure_years derived from naive tenure_days
            if extracted.get("tenure_years") is None and extracted.get("tenure_days"):
                extracted["tenure_years"] = round(extracted["tenure_days"] / 365, 4)
            # maturity_date: use LLM's value as-is (that is the ablation point)

        # ── FIX 1.5: Python decides is_premature by comparing dates ──────────
        # (Ablation 5: when OFF, uses LLM's is_premature flag as-is)
        try:
            if ABLATION_PREMATURE_OVERRIDE:
                if extracted.get("start_date") and extracted.get("tenure_days"):
                    if not extracted.get("maturity_date"):
                        md = (datetime.strptime(extracted["start_date"], "%Y-%m-%d")
                              + timedelta(days=int(extracted["tenure_days"])))
                        extracted["maturity_date"] = md.strftime("%Y-%m-%d")

                    maturity_dt = datetime.strptime(extracted["maturity_date"], "%Y-%m-%d")

                    if extracted.get("withdrawal_date"):
                        wd = datetime.strptime(extracted["withdrawal_date"], "%Y-%m-%d")
                        if wd >= maturity_dt:
                            print(f"Override is_premature=False: {extracted['withdrawal_date']} >= {extracted['maturity_date']}")
                            extracted["is_premature"] = False
                            extracted["holding_days"] = None
                        else:
                            print(f"Confirmed is_premature=True: {extracted['withdrawal_date']} < {extracted['maturity_date']}")
                            extracted["is_premature"] = True
        except Exception as e:
            print(f"Could not evaluate is_premature: {e}")

        # ── FIX 2: Python computes holding_days ───────────────────────────────
        if extracted.get("is_premature") and extracted.get("withdrawal_date") and extracted.get("start_date"):
            try:
                actual = (datetime.strptime(extracted["withdrawal_date"], "%Y-%m-%d") -
                          datetime.strptime(extracted["start_date"], "%Y-%m-%d")).days
                if extracted.get("holding_days") != actual:
                    print(f"Correcting holding_days: {extracted.get('holding_days')} → {actual}")
                    extracted["holding_days"] = actual
            except Exception as e:
                print(f"Could not calculate holding_days: {e}")

        # ── VALIDATE required fields ───────────────────────────────────────────
        if not all([extracted.get("principal"), extracted.get("start_date"), extracted.get("tenure_days")]):
            return {"route": "agent1_error", "error_type": "missing_fields",
                    "final_answer": "I need the FD amount, booking date, and tenure to help you."}

        # ── RATE LOOKUP (Ablation 4) ──────────────────────────────────────────
        citizenship = extracted.get("citizenship_status", "regular")

        if ABLATION_RATE_CARD:
            # ── Structured Excel lookup (default) ─────────────────────────────
            rate_result = get_interest_rate_from_excel(
                booking_date_str   = extracted["start_date"],
                tenure_days        = extracted["tenure_days"],
                citizenship_status = citizenship,
                principal_amount   = extracted["principal"],
                nri_status         = extracted.get("nri_status", "Non NRI"),
            )

            if rate_result["interest_rate"] is None:
                return {"route": "agent1_error", "error_type": "rate_not_found",
                        "final_answer": f"No interest rate found. {rate_result['rate_source']}"}

            extracted["interest_rate_annual_percent"] = rate_result["interest_rate"]
            extracted["rate_source"]                  = rate_result["rate_source"]

            # Effective rate for premature holding period
            if extracted.get("is_premature") and extracted.get("holding_days"):
                eff = get_interest_rate_from_excel(
                    booking_date_str   = extracted["start_date"],
                    tenure_days        = extracted["holding_days"],
                    citizenship_status = citizenship,
                    principal_amount   = extracted["principal"],
                    nri_status         = extracted.get("nri_status", "Non NRI"),
                )
                extracted["effective_rate_for_holding_period"] = (
                    eff["interest_rate"] if eff["interest_rate"] is not None
                    else rate_result["interest_rate"]
                )

        else:
            # ── Ablation 4: LLM reads pre-filtered text rate card ─────────────
            deposit_cat = extracted.get("deposit_amount", "<3CR")
            rate_rows = filter_interest_rates(
                booking_date_str   = extracted["start_date"],
                citizenship_status = citizenship,
                nri_status         = extracted.get("nri_status", "Non NRI"),
                deposit_amount     = deposit_cat,
                # tenure_days NOT passed — LLM does bracket matching
            )

            if not rate_rows:
                return {"route": "agent1_error", "error_type": "rate_not_found",
                        "final_answer": "No rates found for given criteria (ablation 4 mode)."}

            rate_table_text = "Tenure (days) | Rate (% p.a.)\n" + "\n".join(
                f"{row['tenure']} | {row['rate']}%" for row in rate_rows
            )

            rate_pick_prompt = f"""You are a rate lookup assistant.

Given this interest rate table for {citizenship} citizen, {extracted.get('nri_status','Non NRI')}, deposit {deposit_cat}:

{rate_table_text}

The FD tenure is {extracted['tenure_days']} days.

Which row applies? Reply with ONLY the interest rate as a number, e.g. 7.25
Do not write % or any other text. Just the number."""

            rate_resp = client.chat.completions.create(
                model      = get_model("agent1"),
                messages   = [{"role": "user", "content": rate_pick_prompt}],
                max_tokens = 10,
                temperature= 0.0,
            ).choices[0].message.content.strip()

            try:
                llm_rate = round(float(rate_resp.replace("%", "").strip()), 2)
                extracted["interest_rate_annual_percent"] = llm_rate
                extracted["rate_source"] = f"LLM-text-ratecard (ablation4) | tenure={extracted['tenure_days']}d"
            except Exception as e:
                return {"route": "agent1_error", "error_type": "rate_not_found",
                        "final_answer": f"Ablation 4 rate lookup failed: LLM returned '{rate_resp}'. Error: {e}"}

            # Effective rate for premature holding period (ablation 4 path)
            if extracted.get("is_premature") and extracted.get("holding_days"):
                eff_rows = filter_interest_rates(
                    booking_date_str   = extracted["start_date"],
                    citizenship_status = citizenship,
                    nri_status         = extracted.get("nri_status", "Non NRI"),
                    deposit_amount     = deposit_cat,
                )
                if eff_rows:
                    eff_table = "Tenure (days) | Rate (% p.a.)\n" + "\n".join(
                        f"{r['tenure']} | {r['rate']}%" for r in eff_rows
                    )
                    eff_prompt = f"""Given this rate table:
{eff_table}

Holding period: {extracted['holding_days']} days.
Reply with ONLY the applicable rate as a number."""

                    eff_resp = client.chat.completions.create(
                        model      = get_model("agent1"),
                        messages   = [{"role": "user", "content": eff_prompt}],
                        max_tokens = 10,
                        temperature= 0.0,
                    ).choices[0].message.content.strip()

                    try:
                        extracted["effective_rate_for_holding_period"] = round(
                            float(eff_resp.replace("%", "").strip()), 2)
                    except:
                        extracted["effective_rate_for_holding_period"] = extracted["interest_rate_annual_percent"]
                else:
                    extracted["effective_rate_for_holding_period"] = extracted["interest_rate_annual_percent"]

        extracted["tenure_years"] = round(extracted["tenure_days"] / 365, 4)
        return extracted

    except Exception as e:
        print(f"GLOBAL ERROR in run_agent1: {e}")
        return {"route": "agent1_error", "error_type": "exception", "final_answer": str(e)}

In [61]:
# REPLACE WITH:
class RAGEngine:
    def __init__(self, doc_path, chunk_size=800, chunk_overlap=150):
        self.embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

        # Load document — supports both .docx and .pdf
        if doc_path.endswith(".pdf"):
            import pdfplumber
            paragraphs = []
            with pdfplumber.open(doc_path) as pdf:
                for page in pdf.pages:
                    text = page.extract_text()
                    if text:
                        for line in text.split("\n"):
                            line = line.strip()
                            if line:
                                paragraphs.append(line)
        else:
            doc = Document(doc_path)
            paragraphs = [p.text.strip() for p in doc.paragraphs if p.text.strip()]

        # Chunk with overlap
        # First join all paragraphs into full text, then sliding window chunk
        full_text = "\n\n".join(paragraphs)
        self.chunked_docs = []
        start = 0
        while start < len(full_text):
            end = start + chunk_size
            self.chunked_docs.append(full_text[start:end])
            if end >= len(full_text):
                break
            start += chunk_size - chunk_overlap

        # Embed & build FAISS index
        self.doc_embeddings = self.embedding_model.encode(self.chunked_docs)
        self.index = faiss.IndexFlatL2(self.doc_embeddings.shape[1])
        self.index.add(np.array(self.doc_embeddings))

In [62]:
rate_policy_rag = RAGEngine('/content/sample_data/Guidelines_pdf.pdf')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [63]:
def retrieve_rate_policy_context(user_query, top_k=5):
    qvec = rate_policy_rag.embedding_model.encode([user_query])
    D, I = rate_policy_rag.index.search(np.array(qvec), top_k)
    return "\n\n".join(rate_policy_rag.chunked_docs[i] for i in I[0])

In [64]:
def get_interest_rate_from_excel(booking_date_str, tenure_days, citizenship_status,
                                  principal_amount=None, nri_status="Non NRI",
                                  excel_path='/content/sample_data/int_rate_new.xlsx'):
    """
    Lookup interest rate - FLEXIBLE column name handling.
    """
    df = pd.read_excel(excel_path)

    booking_date = datetime.strptime(booking_date_str, "%Y-%m-%d")

    # Convert dates
    df['Start Date'] = pd.to_datetime(df['Start Date'], format='%d-%m-%y', errors='coerce')
    df['End Date'] = pd.to_datetime(df['End Date'], format='%d-%m-%y', errors='coerce')
    df['End Date'] = df['End Date'].apply(lambda x: datetime(2099, 12, 31) if pd.notna(x) and x.year == 1999 else x)

    # Citizenship mapping
    citizen_type = "Senior" if citizenship_status == "senior" else "Regular"

    # Deposit category
    if principal_amount is None or principal_amount < 30000000:
        deposit_category = "<3CR"
    else:
        deposit_category = ">=3CR"

    # 🔥 FIX: Auto-detect deposit amount column name
    deposit_col = None
    for col in ['Deposit amt', 'AMT', 'Deposit_amt', 'Deposit Amount', 'deposit_amt']:
        if col in df.columns:
            deposit_col = col
            break

    if deposit_col is None:
        print(f"⚠️ Available columns: {df.columns.tolist()}")
        return {'interest_rate': None, 'rate_source': "Deposit amount column not found in Excel"}

    # 🔥 FIX: Auto-detect NRI status column name
    nri_col = None
    for col in ['Citizenship status(NRI/NON NRI)', 'NRI Status', 'Citizenship Status', 'citizenship_status']:
        if col in df.columns:
            nri_col = col
            break

    # Filter
    df_filtered = df[
        (df['Start Date'] <= booking_date) &
        (df['End Date'] >= booking_date) &
        (df['Citizen Type'] == citizen_type) &
        (df['Tenure Start'] <= tenure_days) &
        (df['Tenure End'] >= tenure_days)
    ]

    # Apply deposit filter if column exists
    if deposit_col:
        df_filtered = df_filtered[df_filtered[deposit_col] == deposit_category]

    # Apply NRI filter if column exists
    if nri_col:
        df_filtered = df_filtered[df_filtered[nri_col] == nri_status]

    if df_filtered.empty:
        return {'interest_rate': None, 'rate_source': "No rate found for given criteria"}

    df_filtered = df_filtered.sort_values('Priority', ascending=True)
    best_match = df_filtered.iloc[0]

    # Convert rate
    rate_str = str(best_match['Int Rate'])
    if '%' in rate_str:
        rate_float = round(float(rate_str.strip('%')),2)
    else:
        rate_float = round(float(rate_str) * 100,2)

    rate_source = f"Rate card: {best_match['Start Date'].strftime('%d-%b-%Y')} to {best_match['End Date'].strftime('%d-%b-%Y')}"

    return {'interest_rate': rate_float, 'rate_source': rate_source}

In [65]:
def filter_interest_rates(booking_date_str=None, tenure_days=None, citizenship_status=None,
                          nri_status=None, deposit_amount=None,
                          excel_path='/content/sample_data/int_rate_new.xlsx'):
    """
    FIXED: Proper type handling for comparisons.
    """
    df = pd.read_excel(excel_path)

    # Date conversion
    df['Start Date'] = pd.to_datetime(df['Start Date'], format='%d-%b-%y', errors='coerce')
    df['End Date'] = pd.to_datetime(df['End Date'], format='%d-%b-%y', errors='coerce')
    df['End Date'] = df['End Date'].apply(lambda x: datetime(2099, 12, 31) if pd.notna(x) and x.year == 1999 else x)

    # 🔥 FIX: Ensure numeric columns are actually numeric
    df['Tenure Start'] = pd.to_numeric(df['Tenure Start'], errors='coerce')
    df['Tenure End'] = pd.to_numeric(df['Tenure End'], errors='coerce')
    df['Priority'] = pd.to_numeric(df['Priority'], errors='coerce')

    # Default to current date
    if booking_date_str:
        try:
            booking_date = datetime.strptime(booking_date_str, "%Y-%m-%d")
        except:
            booking_date = datetime.now()
    else:
        booking_date = datetime.now()

    # Filter by date
    df_filtered = df[
        (df['Start Date'] <= booking_date) &
        (df['End Date'] >= booking_date)
    ]

    # Auto-detect column names
    deposit_col = None
    for col in ['Deposit amt', 'AMT', 'Deposit_amt', 'Deposit Amount']:
        if col in df.columns:
            deposit_col = col
            break

    nri_col = None
    for col in ['Citizenship status(NRI/NON NRI)', 'NRI Status', 'Citizenship Status']:
        if col in df.columns:
            nri_col = col
            break

    # Apply filters
    if citizenship_status:
        citizen_type = "Senior" if citizenship_status == "senior" else "Regular"
        df_filtered = df_filtered[df_filtered['Citizen Type'] == citizen_type]

    if nri_status and nri_col:
        df_filtered = df_filtered[df_filtered[nri_col] == nri_status]
    elif nri_col:
        df_filtered = df_filtered[df_filtered[nri_col] == "Non NRI"]

    if deposit_amount and deposit_col:
        df_filtered = df_filtered[df_filtered[deposit_col] == deposit_amount]
    elif deposit_col:
        df_filtered = df_filtered[df_filtered[deposit_col] == "<3CR"]

    # 🔥 FIX: Convert tenure_days to int if it's a string
    if tenure_days is not None:
        try:
            tenure_days = int(tenure_days)
        except (ValueError, TypeError):
            print(f"⚠️ Warning: Could not convert tenure_days '{tenure_days}' to int")
            tenure_days = None

    if tenure_days is not None:
        df_filtered = df_filtered[
            (df_filtered['Tenure Start'] <= tenure_days) &
            (df_filtered['Tenure End'] >= tenure_days)
        ]

    if df_filtered.empty:
        return []

    # Return results
    results = []
    for _, row in df_filtered.iterrows():
        rate_str = str(row['Int Rate'])
        rate_value = float(rate_str.rstrip('%')) if '%' in rate_str else float(rate_str) * 100

        result = {
            'rate': rate_value,
            'citizen': row['Citizen Type'],
            'tenure': f"{int(row['Tenure Start'])}-{int(row['Tenure End'])} days",
            'priority': int(row['Priority'])
        }

        if nri_col and nri_col in row:
            result['nri'] = row[nri_col]
        if deposit_col and deposit_col in row:
            result['deposit'] = row[deposit_col]

        results.append(result)

    # Sort by rate descending
    results = sorted(results, key=lambda x: (-x['rate'], x['priority']))
    return results

In [66]:
validation_agent_prompt = """
You are a Validation Agent for FD (Fixed Deposit) queries.

Your job is to check if the user has provided ALL the critical information needed to calculate FD interest.

Critical information required:
1. **FD Amount/Principal** - How much money was deposited (e.g., Rs. 1,00,000, 50000, 1 lakh)
2. **FD Booking/Start Date** - When the FD was booked (e.g., 10-Mar-2025, 15th January 2024, last Monday)
3. **FD Tenure/Duration** - How long the FD is for (e.g., 2 years, 18 months, 400 days)

Optional information (nice to have but not critical):
- Payout type (quarterly, monthly, cumulative) - we can infer this
- Citizenship status (regular/senior) - we can assume regular if not mentioned
- Premature withdrawal date - only needed if breaking FD early

Your task:
1. Analyze the user query
2. Check if ALL THREE critical pieces of information are present
3. Return JSON with validation result

Output format:
{
  "is_valid": true/false,
  "has_principal": true/false,
  "has_start_date": true/false,
  "has_tenure": true/false,
  "missing_fields": ["field1", "field2"],  // empty array if all present
  "extracted_info": {
    "principal_mentioned": "what user said about amount",
    "start_date_mentioned": "what user said about date",
    "tenure_mentioned": "what user said about tenure"
  },
  "user_message": "friendly message asking for missing info (null if valid)"
}

Guidelines:
- Be lenient with date formats - "last week", "10th March", "today" are all valid
- Be lenient with amounts - "1 lakh", "100000", "Rs 50k" are all valid
- Be lenient with tenure - "2 yrs", "24 months", "730 days" are all valid
- If user says "I booked FD" without date, start_date is MISSING
- If user asks about "my FD" without any context, treat as MISSING information
- Interest rate mentioned by user is NOT required (we determine from our rate card)

Examples:

Query: "Booked an fd of 1000rs at 10% interest howmuch income will i get?"
Output:
{
  "is_valid": false,
  "has_principal": true,
  "has_start_date": false,
  "has_tenure": false,
  "missing_fields": ["start_date", "tenure"],
  "extracted_info": {
    "principal_mentioned": "1000rs",
    "start_date_mentioned": null,
    "tenure_mentioned": null
  },
  "user_message": "To calculate your FD interest accurately, I need:\n\n1. **FD booking date** - When did you book this FD? (e.g., 10-Mar-2025, last Monday)\n2. **FD tenure** - For how long is this FD? (e.g., 2 years, 18 months)\n\nExample: 'I booked an FD of Rs. 1,000 on 15-Jan-2025 for 2 years. How much interest will I get?'"
}

Query: "How much interest income I'll get in the first quarter of booking FD on 23Sep2024? FD booking amount 1,00,000, tenure 3 years. I am a senior citizen"
Output:
{
  "is_valid": true,
  "has_principal": true,
  "has_start_date": true,
  "has_tenure": true,
  "missing_fields": [],
  "extracted_info": {
    "principal_mentioned": "1,00,000",
    "start_date_mentioned": "23Sep2024",
    "tenure_mentioned": "3 years"
  },
  "user_message": null
}

Query: "My FD interest seems low"
Output:
{
  "is_valid": false,
  "has_principal": false,
  "has_start_date": false,
  "has_tenure": false,
  "missing_fields": ["principal", "start_date", "tenure"],
  "extracted_info": {
    "principal_mentioned": null,
    "start_date_mentioned": null,
    "tenure_mentioned": null
  },
  "user_message": "To help you understand your FD interest, I need:\n\n1. **FD amount** - How much did you deposit? (e.g., Rs. 1,00,000)\n2. **FD booking date** - When did you book this FD? (e.g., 10-Mar-2025)\n3. **FD tenure** - For how long is this FD? (e.g., 2 years)\n\nExample: 'I booked an FD of Rs. 50,000 on 10-Jan-2025 for 18 months. Why is my interest low?'"
}

Return ONLY valid JSON. No explanations outside the JSON.
"""

In [67]:
def run_validation_agent(user_query):
    """
    Validates user query has all critical info (principal, start_date, tenure).
    Returns dict with is_valid and user_message.
    """
    resp = client.chat.completions.create(
        model=get_model("agent1"),
        messages=[
            {"role": "system", "content": "You are a validation agent. Return ONLY a single valid JSON object. No explanation, no extra text, no markdown."},
            {"role": "user", "content": f"{validation_agent_prompt}\n\nUSER QUERY:\n{user_query}\n\nReturn ONLY the JSON."}
        ],
        temperature=0.0,
        max_tokens=300
    ).choices[0].message.content

    try:
        validation_result = extract_json_from_text(resp)
        return validation_result
    except Exception:
        # If LLM returned verbose/multiple JSON blocks, fall back to Python-only check
        # Simple heuristic: if query has a number, a date-like string, and a duration
        import re
        q = user_query.lower()
        has_amount  = bool(re.search(r'\d[\d,\.]*\s*(rs|lakh|crore|thousand|rupee|₹)', q) or
                          re.search(r'(rs\.?|₹)\s*[\d,]+', q))
        has_date    = bool(re.search(r'\d{1,2}[\-/th|st|nd|rd\s]*(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec|\d{4})', q))
        has_tenure  = bool(re.search(r'\d+\s*(year|month|day|quarter)', q))

        missing = []
        if not has_amount:  missing.append("FD amount")
        if not has_date:    missing.append("booking date")
        if not has_tenure:  missing.append("tenure/duration")

        if missing:
            return {
                "is_valid": False,
                "issues": [f"Missing: {', '.join(missing)}"],
                "missing_fields": missing,
                "invalid_fields": {},
                "user_message": f"To calculate your FD interest, I need: {', '.join(missing)}.\n\nExample: 'I booked an FD of Rs. 1,00,000 on 10-Mar-2025 for 2 years.'"
            }
        return {"is_valid": True, "issues": [], "missing_fields": [], "invalid_fields": {}, "user_message": None}


In [68]:
agent1_prompt = f"""Today's date is {TODAY_STR}.

Use this as "today" whenever the user says "today", "now", "current", "breaking today",
"closing today", "withdrawing today", or does not mention a date.

IMPORTANT FOR PREMATURE:
- If user says "breaking today" / "closing today" / "withdrawing today"
  → withdrawal_date = "{TODAY_ISO}"
  → is_premature = true
- If user says "booked today" / "started today" / does not mention booking date
  → start_date = "{TODAY_ISO}"

You are Agent-1: FD details extractor.

Your responsibilities:

------------------------------------------------------------
1. Extract metadata from the user query:
------------------------------------------------------------
- principal amount
- FD start / booking date
- premature withdrawal date (if mentioned)
- determine is_premature = true/false

- tenure — extract the RAW value and unit ONLY. Python will compute exact dates.
    a) "9 months"         → tenure_value=9,   tenure_unit="months"
    b) "2 years"          → tenure_value=2,   tenure_unit="years"
    c) "1.5 years"        → tenure_value=1.5, tenure_unit="years"
    d) "4 quarters"       → tenure_value=4,   tenure_unit="quarters"
    e) "730 days"         → tenure_value=730, tenure_unit="days"
    f) "1 year 6 months"  → tenure_value=18,  tenure_unit="months"  (convert to total months)
    g) "18 months"        → tenure_value=18,  tenure_unit="months"
    h) maturity date given, no tenure → tenure_value=(maturity-start).days, tenure_unit="days"

    DO NOT compute tenure_days yourself. Just give tenure_value and tenure_unit.
    tenure_days and maturity_date will be computed by Python.

- payout type:

    ============================================================
    HIGHEST PRIORITY RULE - CUMULATIVE OVERRIDE (READ FIRST):
    ============================================================
    If the user mentions ANY of these words → payout_type = "cumulative" ALWAYS:
      * "cumulative"
      * "at maturity"
      * "on maturity"
      * "maturity amount"
      * "quarterly compounding"  (compounding != payout)
      * "cumulative quarterly"   = quarterly compounding, paid at maturity = cumulative
      * "cumulative monthly"     = monthly compounding, paid at maturity   = cumulative

    This rule OVERRIDES all other rules below.
    Even if "quarterly" or "monthly" also appears in the query,
    if "cumulative" is present → payout_type = "cumulative".

    ============================================================
    SECONDARY RULES - apply ONLY if cumulative override does NOT apply:
    ============================================================
    * If user asks "Q1", "Q2", "Q3", "Q4", "first quarter", "second quarter",
      "quarterly interest", "interest each quarter", "quarterly payout",
      "quarterly interest payout"
      → payout_type = "quarterly_payout"

    * If user asks "first month", "second month", "monthly interest",
      "interest each month", "monthly payout"
      → payout_type = "monthly_payout"

    * If user asks "first half year", "half yearly interest",
      "interest every 6 months", "half yearly payout"
      → payout_type = "half_yearly_payout"

    * Default (if nothing specified) → payout_type = "cumulative"

    ============================================================
    DISAMBIGUATION TABLE (memorize these):
    ============================================================
    User says                          | payout_type
    -----------------------------------|-------------------
    "cumulative quarterly"             | cumulative       ✅
    "quarterly compounding"            | cumulative       ✅
    "cumulative FD"                    | cumulative       ✅
    "maturity amount"                  | cumulative       ✅
    "at maturity"                      | cumulative       ✅
    "cumulative monthly"               | cumulative       ✅
    "quarterly payout"                 | quarterly_payout
    "quarterly interest payout"        | quarterly_payout
    "interest each quarter"            | quarterly_payout
    "Q1 interest" / "first quarter"    | quarterly_payout
    "monthly payout"                   | monthly_payout
    "interest each month"              | monthly_payout
    "half yearly payout"               | half_yearly_payout
    "interest every 6 months"          | half_yearly_payout

- citizenship_status = "regular" or "senior"

- nri_status = "NRI" or "Non NRI":
    * If user mentions "NRI" or "non-resident" → nri_status = "NRI"
    * Default (if not mentioned) → nri_status = "Non NRI"
    IMPORTANT: You MUST include nri_status in the output JSON.

- deposit_amount = "<3CR" or ">=3CR":
    * If principal >= 3,00,00,000 (3 crores) → deposit_amount = ">=3CR"
    * If principal < 3,00,00,000 → deposit_amount = "<3CR"
    IMPORTANT: You MUST include deposit_amount in the output JSON.

------------------------------------------------------------
2. CRITICAL: Detect premature withdrawal:
------------------------------------------------------------
- If the user mentions a second date (closing/withdrawing/breaking) → set withdrawal_date to that date
- If no withdrawal mentioned → withdrawal_date = null
- Do NOT set is_premature yourself. Python will decide based on dates.
- holding_days = null always (Python computes this)

------------------------------------------------------------
3. Output format:
------------------------------------------------------------
{{
  "principal": <number>,
  "start_date": "<YYYY-MM-DD>",
  "tenure_value": <number>,
  "tenure_unit": "<days|months|years|quarters>",
  "withdrawal_date": "<YYYY-MM-DD or null>",
  "holding_days": <integer or null>,
  "payout_type": "<cumulative | quarterly_payout | monthly_payout | half_yearly_payout>",
  "is_premature": <true|false>,
  "citizenship_status": "<regular|senior>",
  "nri_status": "<NRI|Non NRI>",
  "deposit_amount": "<<3CR|>=3CR>"
}}

Return ONLY JSON. No explanations. No text before or after the JSON.
Do NOT include tenure_days or tenure_years — Python computes these.
Do NOT calculate interest rates.

------------------------------------------------------------
4. CRITICAL: Calculate deposit_amount automatically:
------------------------------------------------------------
- If principal >= 30000000 (3 crores) → deposit_amount = ">=3CR"
- If principal < 30000000 → deposit_amount = "<3CR"

------------------------------------------------------------
EXAMPLES:
------------------------------------------------------------

Example 1:
Query: "I booked 50000 rs FD for 3 years on 15 jan 2024. I am senior citizen"
Output:
{{
  "principal": 50000,
  "start_date": "2024-01-15",
  "tenure_value": 3,
  "tenure_unit": "years",
  "withdrawal_date": null,
  "holding_days": null,
  "payout_type": "cumulative",
  "is_premature": false,
  "citizenship_status": "senior",
  "nri_status": "Non NRI",
  "deposit_amount": "<3CR"
}}

Example 2:
Query: "Booked 5 crore FD for 2 years on 10-Mar-2025, NRI, senior citizen"
Output:
{{
  "principal": 50000000,
  "start_date": "2025-03-10",
  "tenure_value": 2,
  "tenure_unit": "years",
  "withdrawal_date": null,
  "holding_days": null,
  "payout_type": "cumulative",
  "is_premature": false,
  "citizenship_status": "senior",
  "nri_status": "NRI",
  "deposit_amount": ">=3CR"
}}

Example 3:
Query: "I have FD of 2 lakh booked on 1-Jan-2024 for 5 years"
Output:
{{
  "principal": 200000,
  "start_date": "2024-01-01",
  "tenure_value": 5,
  "tenure_unit": "years",
  "withdrawal_date": null,
  "holding_days": null,
  "payout_type": "cumulative",
  "is_premature": false,
  "citizenship_status": "regular",
  "nri_status": "Non NRI",
  "deposit_amount": "<3CR"
}}

Example 4:
Query: "FD 2 lakh on 5 June 2025 for 5 years, regular citizen, cumulative quarterly. What is maturity amount?"
Output:
{{
  "principal": 200000,
  "start_date": "2025-06-05",
  "tenure_value": 5,
  "tenure_unit": "years",
  "withdrawal_date": null,
  "holding_days": null,
  "payout_type": "cumulative",
  "is_premature": false,
  "citizenship_status": "regular",
  "nri_status": "Non NRI",
  "deposit_amount": "<3CR"
}}

Example 5:
Query: "Booked FD on 23 Sep 2024, amount 1 lakh, 3 years tenure, senior citizen. How much quarterly interest payout will I get?"
Output:
{{
  "principal": 100000,
  "start_date": "2024-09-23",
  "tenure_value": 3,
  "tenure_unit": "years",
  "withdrawal_date": null,
  "holding_days": null,
  "payout_type": "quarterly_payout",
  "is_premature": false,
  "citizenship_status": "senior",
  "nri_status": "Non NRI",
  "deposit_amount": "<3CR"
}}

Example 6:
Query: "I deposited Rs. 5 lakh on 1 Apr 2025 for 2 years with quarterly compounding. What will I get at maturity?"
Output:
{{
  "principal": 500000,
  "start_date": "2025-04-01",
  "tenure_value": 2,
  "tenure_unit": "years",
  "withdrawal_date": null,
  "holding_days": null,
  "payout_type": "cumulative",
  "is_premature": false,
  "citizenship_status": "regular",
  "nri_status": "Non NRI",
  "deposit_amount": "<3CR"
}}

Example 7:
Query: "I booked an FD of Rs. 5,00,000 on 30th November 2023 for 9 months. How much will I receive monthly?"
Output:
{{
  "principal": 500000,
  "start_date": "2023-11-30",
  "tenure_value": 9,
  "tenure_unit": "months",
  "withdrawal_date": null,
  "holding_days": null,
  "payout_type": "monthly_payout",
  "is_premature": false,
  "citizenship_status": "regular",
  "nri_status": "Non NRI",
  "deposit_amount": "<3CR"
}}

Example 8:
Query: "I closed my FD early. Booked Rs. 1,50,000 on 10 March 2023 for 2 years, closed on 5 December 2024."
Output:
{{
  "principal": 150000,
  "start_date": "2023-03-10",
  "tenure_value": 2,
  "tenure_unit": "years",
  "withdrawal_date": "2024-12-05",
  "holding_days": null,
  "payout_type": "cumulative",
  "is_premature": true,
  "citizenship_status": "regular",
  "nri_status": "Non NRI",
  "deposit_amount": "<3CR"
}}

Example 9:
Query: "I closed my FD early. Booked Rs. 2,00,000 on 1 Jan 2024 for 3 years, closed on 15 Aug 2024."
Output:
{{
  "principal": 200000,
  "start_date": "2024-01-01",
  "tenure_value": 3,
  "tenure_unit": "years",
  "withdrawal_date": "2024-08-15",
  "holding_days": null,
  "payout_type": "cumulative",
  "is_premature": true,
  "citizenship_status": "regular",
  "nri_status": "Non NRI",
  "deposit_amount": "<3CR"
}}

Example 10:
Query: "FD of 3 lakh on 31 Jan 2024 for 18 months, monthly payout. How much per month?"
Output:
{{
  "principal": 300000,
  "start_date": "2024-01-31",
  "tenure_value": 18,
  "tenure_unit": "months",
  "withdrawal_date": null,
  "holding_days": null,
  "payout_type": "monthly_payout",
  "is_premature": false,
  "citizenship_status": "regular",
  "nri_status": "Non NRI",
  "deposit_amount": "<3CR"
}}

Example 11:
Query: "Booked FD of 1 lakh on 29 Feb 2024 for 2 years. What is the maturity amount?"
Output:
{{
  "principal": 100000,
  "start_date": "2024-02-29",
  "tenure_value": 2,
  "tenure_unit": "years",
  "withdrawal_date": null,
  "holding_days": null,
  "payout_type": "cumulative",
  "is_premature": false,
  "citizenship_status": "regular",
  "nri_status": "Non NRI",
  "deposit_amount": "<3CR"
}}

Example 12:
Query: "I have an FD of 2 lakh for 4 quarters starting 1 Oct 2024. Quarterly payout."
Output:
{{
  "principal": 200000,
  "start_date": "2024-10-01",
  "tenure_value": 4,
  "tenure_unit": "quarters",
  "withdrawal_date": null,
  "holding_days": null,
  "payout_type": "quarterly_payout",
  "is_premature": false,
  "citizenship_status": "regular",
  "nri_status": "Non NRI",
  "deposit_amount": "<3CR"
}}
"""

In [69]:
# =========================================================
# clean_unicode_text FUNCTION
# =========================================================

def clean_unicode_text(text):
    """
    Removes problematic Unicode characters from LLM output.
    Replaces them with plain ASCII equivalents.
    """

    # Replace Unicode rupee symbol with Rs.
    text = text.replace('₹', 'Rs. ')

    # Replace multiplication sign with x
    text = text.replace('×', 'x')
    text = text.replace('\u00d7', 'x')  # Explicit Unicode for multiplication

    # Replace various Unicode dashes and spaces with regular ones
    text = text.replace('\u2013', '-')  # en dash
    text = text.replace('\u2014', '-')  # em dash
    text = text.replace('\u2011', '-')  # non-breaking hyphen
    text = text.replace('\u202f', ' ')  # narrow no-break space
    text = text.replace('\u2019', "'")  # right single quotation mark
    text = text.replace('\u2018', "'")  # left single quotation mark
    text = text.replace('\u201c', '"')  # left double quotation mark
    text = text.replace('\u201d', '"')  # right double quotation mark

    # Replace other mathematical symbols
    text = text.replace('\u00f7', '/')  # division sign
    text = text.replace('\u2212', '-')  # minus sign

    # Remove zero-width spaces and other invisible characters
    text = text.replace('\u200b', '')  # zero-width space
    text = text.replace('\ufeff', '')  # zero-width no-break space

    return text

In [70]:
# =========================================================
# AGENT1 OUTPUT VALIDATOR
# =========================================================

agent1_validator_prompt = """
You are an Agent1 Output Validator.

You receive the extracted FD details from Agent-1 and must verify that all critical fields are present and valid.

Critical fields that MUST be present:
1. principal     - Must be a positive number
2. start_date    - Must be a valid date in YYYY-MM-DD format
3. tenure_value  - Must be a positive number (the raw tenure amount e.g. 2, 9, 4)
4. tenure_unit   - Must be one of: "days", "months", "years", "quarters"

NOTE: tenure_days is NO LONGER in Agent-1 output. Check tenure_value + tenure_unit instead.

Your task:
Check if Agent1 successfully extracted all these fields with valid values.

Output format (return ONLY this JSON, nothing else):
{
  "is_valid": true/false,
  "issues": [],
  "missing_fields": [],
  "invalid_fields": {},
  "user_message": "friendly message if invalid, null if valid"
}

Examples:

Agent1 Output: {"principal": 100000, "start_date": "2025-03-10", "tenure_value": 2, "tenure_unit": "years"}
Your Output:
{
  "is_valid": true,
  "issues": [],
  "missing_fields": [],
  "invalid_fields": {},
  "user_message": null
}

Agent1 Output: {"principal": 200000, "start_date": "2025-01-10", "tenure_value": 4, "tenure_unit": "quarters"}
Your Output:
{
  "is_valid": true,
  "issues": [],
  "missing_fields": [],
  "invalid_fields": {},
  "user_message": null
}

Agent1 Output: {"principal": null, "start_date": "2025-03-10", "tenure_value": 9, "tenure_unit": "months"}
Your Output:
{
  "is_valid": false,
  "issues": ["Principal amount is missing"],
  "missing_fields": ["principal"],
  "invalid_fields": {},
  "user_message": "I couldn't determine the FD amount from your query. Please specify how much you deposited.\n\nExample: 'I booked an FD of Rs. 1,00,000 on 10-Mar-2025 for 2 years.'"
}

Agent1 Output: {"principal": 100000, "start_date": null, "tenure_value": 2, "tenure_unit": "years"}
Your Output:
{
  "is_valid": false,
  "issues": ["Start date is missing"],
  "missing_fields": ["start_date"],
  "invalid_fields": {},
  "user_message": "I couldn't determine when you booked the FD. Please specify the booking date.\n\nExample: 'I booked an FD on 15-Jan-2025 for 2 years.'"
}

Agent1 Output: {"principal": 100000, "start_date": "2025-03-10", "tenure_value": null, "tenure_unit": null}
Your Output:
{
  "is_valid": false,
  "issues": ["Tenure is missing"],
  "missing_fields": ["tenure_value", "tenure_unit"],
  "invalid_fields": {},
  "user_message": "I couldn't determine the FD tenure. Please specify the duration.\n\nExample: 'I booked an FD on 15-Jan-2025 for 2 years.'"
}

Return ONLY the JSON object. No explanation. No extra text.
"""

In [71]:
# =========================================================
# Pure Python validation — no LLM call needed.
# =========================================================

def validate_agent1_output_with_llm(agent1_output, user_query):
    """
    Validates Agent-1 output fields in pure Python.
    No LLM call — we have the data, just check it.

    Required fields (new schema):
        principal    : positive number
        start_date   : non-null string
        tenure_value : positive number   ← new
        tenure_unit  : valid string      ← new
        tenure_days  : positive int      ← computed by tenure engine

    Returns dict matching old LLM schema so pipeline needs no changes.
    """
    # If agent1 itself returned an error route, pass that through
    if isinstance(agent1_output, dict) and agent1_output.get("route"):
        return {
            "is_valid": False,
            "issues": [agent1_output.get("final_answer", "Agent-1 error")],
            "missing_fields": agent1_output.get("validation_details", {}).get("missing_fields", []),
            "invalid_fields": {},
            "user_message": agent1_output.get("final_answer", "I couldn't extract your FD details.")
        }

    issues        = []
    missing       = []
    invalid       = {}

    # ── principal ─────────────────────────────────────────────────────────────
    principal = agent1_output.get("principal")
    if not principal:
        missing.append("principal")
        issues.append("FD amount is missing")
    elif float(principal) <= 0:
        invalid["principal"] = "Amount must be positive"
        issues.append("FD amount must be positive")

    # ── start_date ────────────────────────────────────────────────────────────
    start_date = agent1_output.get("start_date")
    if not start_date:
        missing.append("start_date")
        issues.append("Booking date is missing")

    # ── tenure — check tenure_days (set by tenure engine) ─────────────────────
    # tenure_days is the authoritative computed field; if it's present and > 0 we're good.
    # Also accept tenure_value+tenure_unit as sufficient (tenure_days may be set slightly later).
    tenure_days  = agent1_output.get("tenure_days")
    tenure_value = agent1_output.get("tenure_value")
    tenure_unit  = agent1_output.get("tenure_unit")
    valid_units  = {"days", "months", "years", "quarters"}

    if tenure_days and int(tenure_days) > 0:
        pass  # tenure engine ran successfully — all good
    elif tenure_value and tenure_unit and str(tenure_unit).lower() in valid_units:
        pass  # tenure engine hasn't run yet but inputs are valid — will compute downstream
    else:
        missing.append("tenure_value")
        missing.append("tenure_unit")
        issues.append("Tenure is missing or invalid")

    # ── interest_rate — warn but don't block ──────────────────────────────────
    rate = agent1_output.get("interest_rate_annual_percent")
    if rate is None:
        # Rate lookup failed — this produces its own error in run_agent1, don't double-fail
        pass

    if issues:
        parts = []
        if "principal" in missing:
            parts.append("FD amount (e.g., Rs. 1,00,000)")
        if "start_date" in missing:
            parts.append("booking date (e.g., 10-Mar-2025)")
        if "tenure_value" in missing or "tenure_unit" in missing:
            parts.append("tenure (e.g., 2 years / 9 months / 4 quarters)")

        user_msg = (
            f"To calculate your FD, I need: {', '.join(parts)}.\n\n"
            f"Example: 'I booked an FD of Rs. 1,00,000 on 15-Jan-2025 for 2 years.'"
            if parts else issues[0]
        )

        return {
            "is_valid": False,
            "issues": issues,
            "missing_fields": missing,
            "invalid_fields": invalid,
            "user_message": user_msg
        }

    return {
        "is_valid": True,
        "issues": [],
        "missing_fields": [],
        "invalid_fields": {},
        "user_message": None
    }

In [72]:
# =========================================================
# 🔹 AGENT RUNNER — Switch between Groq and OpenRouter
# =========================================================
# ┌─────────────────────────────────────────────────────┐
# │  USE_OPENROUTER = 0  →  Groq (default)              │
# │  USE_OPENROUTER = 1  →  OpenRouter (fallback)        │
# └─────────────────────────────────────────────────────┘
USE_OPENROUTER = 1   # ← change to 0 for Groq, 1 for OpenRouter

from google.colab import userdata

if USE_OPENROUTER == 0:
    client = Groq(api_key=userdata.get('GROQ_API_KEY'))
    print("✅ Using Groq")
else:
    client = OpenAI(
        api_key=userdata.get('Open_router_key'),#"",
        base_url="https://openrouter.ai/api/v1",
    )
    print("✅ Using OpenRouter")

✅ Using OpenRouter


In [73]:
def sanitize_query_for_llm(user_query):
    """Replace tokens that cause content filtering in gpt-oss-120b."""
    s = user_query
    s = s.replace("₹", "Rs.")
    s = s.replace("\u20b9", "Rs.")
    s = s.replace("broke my",  "closed my")
    s = s.replace("break my",  "close my")
    s = s.replace("broken my", "closed my")
    s = s.replace("I broke",   "I closed")
    s = s.replace("broke the", "closed the")
    return s


In [74]:
# =========================================================
# 🔹 LOAD TEST QUERIES
# =========================================================
golden = pd.read_csv("/content/sample_data/benchmark_demo.csv")
queries = golden["Queries"].tolist()

In [75]:
router_prompt = """
You are the ROUTER agent.

Your job is to decide which workflow the user's question needs.

------------------------------------------------------------
WORKFLOW A → CALCULATION
(Agent1 → Agent2 → Agent3 or Agent4)
------------------------------------------------------------
Choose CALCULATION when the user asks anything that requires
numeric reasoning, date math, or amount comparison, including:

- interest amount
- maturity amount
- interest received
- "I got only …"
- "I received only …"
- "why did I get less interest"
- "less interest"
- "interest is low"
- "interest mismatch"
- "amount mismatch"
- payout mismatch
- payout amount
- "how much will I get"
- quarterly / monthly / half-yearly interest
- interest on premature withdrawal
- premature settlement amount
- comparing interest
- date-based calculations
- checking if bank paid correct interest
- TDS amount or net interest
- any question involving rupee values or numerical judgement

------------------------------------------------------------
WORKFLOW B → RAG_RULES
Use this when the user asks for:
- FD policies and rules (conceptual)
- premature withdrawal rules (conceptual, no calculation)
- penalty rules (conceptual)
- TDS explanation or policy
- Form 15G / 15H details
- general banking rules
- "what happens if" style conceptual questions
- purely informational answers with no numbers
- eligibility rules (who qualifies for what benefit)
- rules about minimum / maximum tenure
- what rate POLICY applies to overdue / unclaimed / deceased deposits
- whether a penalty is waived in a specific situation

------------------------------------------------------------
WORKFLOW C → RATE_QUERY
Use this when the user asks about INTEREST RATES from the rate card:

TRIGGER PHRASES:
- "what is the interest rate"
- "what was the interest rate"
- "what rate will I get"
- "current interest rate"
- "maximum interest rate"
- "highest rate"
- "best rate"
- "which tenure gives max/best/highest rate"
- "rate for X years"
- "rate 2 years ago"
- "applicable rate"
- "rate at that time"
- "compare rates"
- "rate difference between"
- "what's the rate for senior vs regular"
- "NRI rates"
- "rates for 3 crore FD"

CRITICAL DISTINCTIONS:
- "What rate will I get?" → RATE_QUERY (asking about rate card)
- "How much interest will I get?" → CALCULATION (asking for amount)
- "I booked FD 2 years ago, what was the rate?" → RATE_QUERY
- "I booked FD 2 years ago, how much interest?" → CALCULATION
- "Can I get interest for 5 days?" → RAG_RULES (policy: <7 days = zero interest)
- "What rate applies to my unclaimed / overdue FD?" → RAG_RULES (rate policy rule, not rate card lookup)
- "Can I negotiate the interest rate?" → RAG_RULES (negotiation policy, not rate card)

EXAMPLES:
✓ RATE_QUERY: "What's the max interest rate for senior citizens?"
✓ RATE_QUERY: "I want to book FD today for 5 years, what rate will I get?"
✓ RATE_QUERY: "What was the rate in Jan 2023 for 3-year FD?"
✓ RATE_QUERY: "Compare rates for regular vs senior citizen"
✗ CALCULATION: "I booked 1 lakh FD, how much interest will I get?"
✗ RAG_RULES: "What is the TDS policy?"

------------------------------------------------------------
Return STRICT JSON ONLY:
Return ONLY a valid JSON object.
Start your response with { and end with }.
No text before or after the JSON.
{
  "route": "calculation" | "rag_rules" | "rate_query",
  "reason": "<one sentence why>"
}
"""

In [76]:
def run_router(user_query):
    resp = client.chat.completions.create(
        model=get_model("router"),  # ← ADD THIS
        messages=[
            {"role": "system", "content": router_prompt},
            {"role": "user", "content": user_query}
        ]
    ).choices[0].message.content

    return extract_json_from_text(resp)

In [77]:
rag_rules_prompt = """
You are Agent-RULES: an FD policy and rule answerer.

You must:
- Use only the provided context
- Give rule-based, explanatory answers
- Do NOT calculate interest
- Do NOT create formulas
- Do NOT assume principal or tenure unless stated
- Use ONLY normal ASCII characters in the output:
  * No unicode escape sequences
  * No non-breaking spaces
  * No narrow spaces
  * No em-dashes or fancy hyphens
  * Use plain text characters: %, -, :, ., /
  * Output clean, user-friendly text only

You must answer questions like:
- What is the current FD rate?
- What is the senior citizen rate?
- What are special tenure options?
- What happens on premature withdrawal?
- What is the penalty?
- What is the TDS rule?
- When is quarterly interest credited?
- Is Form 15H required?

Return a short, clear answer — NO JSON.
"""

In [78]:
def clean_text(s):
    if not isinstance(s, str):
        return s
    replacements = {
        "\u202f": " ",
        "\u2009": " ",
        "\u200A": " ",
        "\u2011": "-",
        "\u2013": "-",
        "\u2014": "-",
        "\u00A0": " ",
        "\u2212": "-",
        "\u20b9": "Rs.",   # ₹ symbol → Rs.
        "\u00e2\u0082\u00b9": "Rs.",  # â‚¹ (mojibake of ₹) → Rs.
    }

    for bad, good in replacements.items():
        s = s.replace(bad, good)
    return s


In [79]:
def run_rag_rules(user_query):
    context = retrieve_rate_policy_context(user_query)

    prompt = f"CONTEXT:\n{context}\n\nQUESTION:\n{user_query}\n\nAnswer clearly."

    resp = client.chat.completions.create(
        model=get_model("rag_rules"),  # ← ADD THIS
        messages=[
            {"role": "system", "content": rag_rules_prompt},
            {"role": "user", "content": prompt}
        ]
    ).choices[0].message.content

    return clean_text(resp.strip())

In [80]:
# SMART CATEGORY EXPANSION - run_agent_rate_query v4 (concise output)
# Shows ALL relevant sub-category combinations based on what user specifies

def run_agent_rate_query(user_query):
    """
    SMART CATEGORY EXPANSION:
    - User specifies NOTHING → Show all 8 combinations
    - User specifies SENIOR → Show 4 combinations (senior × NRI/Non-NRI × <3CR/>=3CR)
    - User specifies SENIOR + >=3CR → Show 2 combinations (senior, >=3CR × NRI/Non-NRI)
    - User specifies ALL 3 → Show 1 specific result

    Always expand the UNSPECIFIED dimensions for "best" queries!
    """
    try:
        # 1️⃣ LLM CALL #1: Extract filters with retry
        filter_prompt = f"""Analyze this FD interest rate query: "{user_query}"

Extract filter parameters. If user doesn't specify something, return null.

YOU MUST RETURN ONLY VALID JSON. NO OTHER TEXT. NO EXPLANATIONS.

JSON FORMAT:
{{
  "citizenship": "senior" | "regular" | null,
  "nri_status": "NRI" | "Non NRI" | null,
  "deposit_amount": "<3CR" | ">=3CR" | null,
  "tenure_days": <number> | null,
  "latest_period_only": true | false,
  "query_type": "best_rate" | "best_tenure" | "specific_rate" | "comparison"
}}

INTERPRETATION RULES:
1. "best rate" / "max rate" / "highest rate" → query_type: "best_rate"
2. "best tenure" / "which tenure" / "what duration" → query_type: "best_tenure"
3. "rate for X" (specific lookup) → query_type: "specific_rate"
4. If user specifies senior/regular → set citizenship, else null
5. If user specifies NRI/Non-NRI → set nri_status, else null
6. If user mentions amount > 3 crores → deposit_amount: ">=3CR", else null

TENURE CONVERSION:
- "1 year" = 365, "2 years" = 730, "3 years" = 1095, "5 years" = 1825

EXAMPLES:
Query: "What tenure gives best rate?"
JSON: {{"query_type": "best_tenure", "citizenship": null, "nri_status": null, "deposit_amount": null, "tenure_days": null, "latest_period_only": true}}

Query: "Best tenure for senior citizens?"
JSON: {{"query_type": "best_tenure", "citizenship": "senior", "nri_status": null, "deposit_amount": null, "tenure_days": null, "latest_period_only": true}}

Query: "Senior citizen with 5 crore, best rate?"
JSON: {{"query_type": "best_rate", "citizenship": "senior", "nri_status": null, "deposit_amount": ">=3CR", "tenure_days": null, "latest_period_only": true}}

Query: "NRI rates?"
JSON: {{"query_type": "specific_rate", "citizenship": null, "nri_status": "NRI", "deposit_amount": null, "tenure_days": null, "latest_period_only": true}}

CRITICAL: Return ONLY the JSON object. Start with {{ and end with }}. Nothing else."""

        max_retries = 2
        filters = None

        for attempt in range(max_retries):
            try:
                filter_resp = client.chat.completions.create(
                    model=get_model("agent_rate_query"),
                    messages=[{"role": "user", "content": filter_prompt}],
                    temperature=0.1,
                    max_tokens=350
                ).choices[0].message.content

                filters = extract_json_from_text(filter_resp)
                break

            except Exception as e:
                if attempt == max_retries - 1:
                    query_lower = user_query.lower()
                    if any(word in query_lower for word in ['tenure', 'duration', 'period', 'how long']):
                        query_type = 'best_tenure'
                    elif any(word in query_lower for word in ['best', 'max', 'highest', 'top']):
                        query_type = 'best_rate'
                    else:
                        query_type = 'specific_rate'

                    filters = {
                        "citizenship": None,
                        "nri_status": None,
                        "deposit_amount": None,
                        "tenure_days": None,
                        "latest_period_only": True,
                        "query_type": query_type
                    }

        if not filters:
            filters = {"latest_period_only": True, "query_type": "best_rate"}

        # Convert tenure_days to int
        if 'tenure_days' in filters and filters['tenure_days'] is not None:
            try:
                filters['tenure_days'] = int(filters['tenure_days'])
            except:
                filters['tenure_days'] = None

        # 2️⃣ SMART LOGIC: Determine which dimensions to expand
        booking_date_str = None
        if filters.get('latest_period_only', True):
            booking_date_str = datetime.now().strftime("%Y-%m-%d")

        query_type = filters.get('query_type', 'best_rate')

        # Expand unspecified dimensions for "best" queries
        if query_type in ['best_rate', 'best_tenure']:
            citizenship_values = [filters['citizenship']] if filters.get('citizenship') else ['regular', 'senior']
            nri_values         = [filters['nri_status']]  if filters.get('nri_status')  else ['Non NRI', 'NRI']
            deposit_values     = [filters['deposit_amount']] if filters.get('deposit_amount') else ['<3CR', '>=3CR']

            all_best_rates = []

            for citizenship in citizenship_values:
                for nri_status in nri_values:
                    for deposit_amount in deposit_values:
                        category_rates = filter_interest_rates(
                            booking_date_str=booking_date_str,
                            citizenship_status=citizenship,
                            nri_status=nri_status,
                            deposit_amount=deposit_amount,
                            tenure_days=filters.get('tenure_days')
                        )

                        if category_rates:
                            best_rate = category_rates[0]  # Already sorted by rate descending
                            all_best_rates.append({
                                'rate':       best_rate['rate'],
                                'citizen':    best_rate['citizen'],
                                'tenure':     best_rate['tenure'],
                                'tenure_days': best_rate.get('tenure_days', 'N/A'),
                                'nri':        nri_status,
                                'deposit':    deposit_amount,
                                'priority':   best_rate['priority'],
                                'category':   f"{citizenship}|{nri_status}|{deposit_amount}"
                            })

            if not all_best_rates:
                return {
                    'route': 'rate_query',
                    'final_answer': "No interest rates found in our rate card for the specified criteria."
                }

            all_best_rates.sort(key=lambda x: x['rate'], reverse=True)
            print(f"📊 Found best rates for {len(all_best_rates)} categories")
            rates = all_best_rates

        else:
            # Original behavior for specific_rate queries
            rates = filter_interest_rates(
                booking_date_str=booking_date_str,
                citizenship_status=filters.get('citizenship'),
                nri_status=filters.get('nri_status'),
                deposit_amount=filters.get('deposit_amount'),
                tenure_days=filters.get('tenure_days')
            )

            if not rates:
                return {
                    'route': 'rate_query',
                    'final_answer': "No matching interest rates found in our rate card for the specified criteria."
                }

            print(f"📊 Found {len(rates)} matching rates")

        # 3️⃣ LLM CALL #2: Format response — concise, table-only output
        rate_summary = []
        for r in rates[:20]:  # Safety limit
            rate_summary.append({
                'rate':       r['rate'],
                'citizen':    r['citizen'],
                'tenure':     r['tenure'],
                'nri':        r.get('nri', 'Non NRI'),
                'deposit':    r.get('deposit', '<3CR'),
                'priority':   r['priority']
            })

        rate_data   = json.dumps(rate_summary, indent=2)
        num_results = len(rate_summary)

        # ── Pick specific_instruction based on scenario ───────────────────────
        if num_results >= 4 and query_type == 'best_tenure':
            specific_instruction = f"""
🎯 TASK: Show the best tenure and rate for each of the {num_results} customer categories.

OUTPUT: A single clean markdown table. Nothing else — no steps, no analysis, no extra headings.

| Customer Type | NRI Status | Deposit Amount | Best Tenure | Rate |
|---------------|------------|----------------|-------------|------|
(fill all {num_results} rows from the data)

After the table: one sentence only — the single most useful insight (e.g. which category gets the highest rate and what tenure).
"""

        elif num_results >= 4 and query_type == 'best_rate':
            specific_instruction = f"""
🎯 TASK: Show the best rate for each of the {num_results} customer categories.

OUTPUT: A single clean markdown table. No steps, no analysis, no extra headings.

| Customer Type | NRI Status | Deposit Amount | Tenure      | Rate |
|---------------|------------|----------------|-------------|------|
(fill all {num_results} rows, highest rate first)

After the table: one sentence only — the single most useful insight.
"""

        elif num_results == 2:
            specific_instruction = """
🎯 TASK: Compare these 2 categories.

Format (2 lines only):
1. [Category 1]: [tenure] at [rate]%
2. [Category 2]: [tenure] at [rate]%

One line: brief note on the difference.
"""

        else:
            specific_instruction = """
🎯 TASK: State the best tenure/rate for this category in one sentence.

Format: "The best tenure is [X days / Y years] at [Z]% for [category details]."
"""

        # ── Build final prompt ────────────────────────────────────────────────
        response_prompt = f"""USER QUERY: {user_query}

RATE DATA ({num_results} results, sorted by rate descending):
{rate_data}

{specific_instruction}

STRICT RULES:
- Use ONLY the data provided. No external knowledge.
- Output ONLY what the task asks — table + one insight line (or as specified above).
- NO step-by-step reasoning. NO ## headings. NO analysis paragraphs.
- NO restating the question. Start directly with the table or answer."""

        response = client.chat.completions.create(
            model=get_model("agent_rate_query"),
            messages=[{"role": "user", "content": response_prompt}],
            temperature=0.1,
            max_tokens=600
        ).choices[0].message.content

        cleaned = clean_text(response.strip())

        # Fallback if LLM response too short
        if not cleaned or len(cleaned) < 20:
            if num_results >= 4:
                lines = [f"**Best Rates by Category ({num_results} options):**\n"]
                for r in rate_summary[:8]:
                    lines.append(f"• {r['citizen']} | {r['nri']} | {r['deposit']}: {r['rate']}% for {r['tenure']}")
                return {
                    'route': 'rate_query',
                    'final_answer': '\n'.join(lines)
                }
            else:
                top = rates[0]
                return {
                    'route': 'rate_query',
                    'final_answer': f"Best option: {top['rate']}% for {top['tenure']} ({top['citizen']} citizens, {top.get('nri', 'Non-NRI')}, {top.get('deposit', '<3CR')} deposits)"
                }

        return {'route': 'rate_query', 'final_answer': cleaned}

    except Exception as e:
        print(f"❌ Rate Query Error: {e}")
        import traceback
        traceback.print_exc()

        return {
            'route': 'rate_query_error',
            'final_answer': "I encountered an error while retrieving interest rates. Please try rephrasing your question. For example:\n- 'What is the best interest rate for senior citizens?'\n- 'Which tenure offers the highest rate?'\n- 'Show me rates for 2-year FD'"
        }

In [81]:
agent2_prompt = f"""Today's date is {TODAY_STR}.
You are Agent-2: the FD CALCULATION PLANNER.

You read:
1) FD JSON extracted by Agent-1
2) The original user question

You decide WHAT calculation is needed.
You do NOT calculate interest. Just classify.

CRITICAL RULES FOR PERIODICITY:

Map payout_type → calc_mode + periodicity:

1. payout_type = "cumulative"
       calc_mode = "cumulative"
       periodicity = "on_maturity"
       internal_compounding = "quarterly"

2. payout_type = "quarterly_payout"
       calc_mode = "periodic_payout"
       periodicity = "quarterly"
       internal_compounding = null

3. payout_type = "monthly_payout"
       calc_mode = "periodic_payout"
       periodicity = "monthly"
       internal_compounding = null

4. payout_type = "half_yearly_payout"
       calc_mode = "periodic_payout"
       periodicity = "half_yearly"
       internal_compounding = null

IMPORTANT: half_yearly_payout is SIMPLE INTEREST per half-year period.
NO internal quarterly compounding. Same formula as monthly/quarterly payout:
Interest = P x (r/100) x (days / rolling_year_days)

QUESTION SCOPE - CRITICAL LOGIC:

- If the user asks for "first quarter/month/half-year" OR "Q1" OR "first period":
      question_scope = "specific_period"
      requested_period_index = 1

- If the user asks for "second quarter" OR "Q2":
      question_scope = "specific_period"
      requested_period_index = 2

- If the user asks for "third quarter" OR "Q3":
      question_scope = "specific_period"
      requested_period_index = 3

- If the user asks for "fourth quarter" OR "Q4":
      question_scope = "specific_period"
      requested_period_index = 4

- If the user asks for full maturity interest or amount (no specific period mentioned):
      question_scope = "full_tenure"
      requested_period_index = null

- If the user asks explicitly "until" a date, or for premature withdrawal:
      question_scope = "until_date"
      until_date = that date in YYYY-MM-DD
      requested_period_index = null

IMPORTANT:
- For periodic_payout FDs, ALWAYS calculate ALL periods (full schedule)
- Even if user asks only for Q1, we need the full schedule to answer Q2/Q3/Q4 later
- The final answer will filter to show only the requested period

SPECIAL CASE: if Agent-1 has is_premature = true:
- calc_mode = "premature"
- question_scope = "until_date"
- until_date = withdrawal_date from Agent-1
- periodicity should still reflect payout_type
  (on_maturity for cumulative, quarterly/monthly/half_yearly for payout FDs).

OUTPUT FORMAT (STRICT JSON):

{{
  "calc_mode": "cumulative" | "periodic_payout" | "premature",
  "periodicity": "on_maturity" | "quarterly" | "monthly" | "half_yearly",
  "internal_compounding": "quarterly" | null,
  "question_scope": "specific_period" | "full_tenure" | "until_date",
  "requested_period_index": <integer or null>,
  "until_date": "<YYYY-MM-DD or empty>",
  "needs_schedule": true,
  "needs_total_interest": true | false,
  "needs_per_period_interest": true,
  "notes": "<short natural language description>"
}}

Return ONLY JSON.
"""

In [82]:
def run_agent2(agent1_json, user_query):
    planner_input = json.dumps(agent1_json, ensure_ascii=False)
    resp = client.chat.completions.create(
        model=get_model("agent2"),  # ← ADD THIS
        messages=[
            {
                "role": "system",
                "content": agent2_prompt
            },
            {
                "role": "user",
                "content": f"FD JSON from Agent 1:\n{planner_input}\n\nOriginal user query:\n{user_query}\n\nReturn ONLY the JSON plan."
            }
        ]
    ).choices[0].message.content

    return extract_json_from_text(resp)

In [83]:
import calendar
from dateutil.relativedelta import relativedelta
from datetime import datetime, date, timedelta


def eom_snap(d):
    """
    Snap a date to the last day of its month if it overshoots.
    e.g. Feb has 28 days in non-leap → any day > 28 becomes 28.
    """
    last_day = calendar.monthrange(d.year, d.month)[1]
    return d.replace(day=min(d.day, last_day))


def build_schedule(start_date_str, tenure_years, periodicity,
                   until_date_str=None, tenure_days=None,
                   allow_partial_final=True):
    """
    Build interest payout periods.

    KEY FIX: payout dates are always computed as:
        start_date + n * months_per_period   (anchor-based, NOT chained)
    then snapped to EOM via eom_snap().

    This prevents date drift — e.g. 31-Mar monthly stays
    Apr30, May31, Jun30, Jul31 ... not Apr30, May30, Jun30, Jul30.
    And 30-Nov quarterly → Feb29(leap)/Feb28, May30, Aug30, Nov30
    (not Feb29→May29 drift from chaining).

    allow_partial_final=True  → normal FD maturity → include partial last period.
    allow_partial_final=False → premature FD       → exclude partial last period.
    """

    start_date = datetime.strptime(start_date_str, "%Y-%m-%d").date()
    periods = []

    # ── Determine end limit ───────────────────────────────────────────────────
    if until_date_str:
        end_limit = datetime.strptime(until_date_str, "%Y-%m-%d").date()
    elif tenure_days is not None:
        end_limit = start_date + timedelta(days=tenure_days)
    else:
        full_years = int(tenure_years)
        fractional_years = tenure_years - full_years
        additional_months = int(round(fractional_years * 12))
        end_limit = start_date + relativedelta(years=full_years, months=additional_months)

    # ── on_maturity: single period ────────────────────────────────────────────
    if periodicity == "on_maturity":
        periods.append((start_date, end_limit))
        return periods

    # ── Map periodicity to month increment ────────────────────────────────────
    if periodicity == "quarterly":
        months_per_period = 3
    elif periodicity == "monthly":
        months_per_period = 1
    elif periodicity == "half_yearly":
        months_per_period = 6
    else:
        raise ValueError(f"Invalid periodicity: {periodicity}")

    # ── Build periods (anchor-based) ──────────────────────────────────────────
    n = 1
    current_start = start_date

    while True:
        # Always anchor from original start_date + n full periods, then EOM snap
        naive_end = start_date + relativedelta(months=months_per_period * n)
        period_end = eom_snap(naive_end)

        if period_end > end_limit:
            # Next payout overshoots maturity date
            if allow_partial_final:
                # Normal maturity → include partial final period up to end_limit
                periods.append((current_start, end_limit))
            # Premature FD → stop here, do not include partial period
            break

        periods.append((current_start, period_end))
        current_start = period_end
        n += 1

        if current_start >= end_limit:
            break

    return periods

In [84]:
def interest_for_period(P, r_percent, period_start, period_end):
    """
    Compute interest for one period.
    ABLATION_ROLLING_YEAR=True  → exact rolling 365/366 days
    ABLATION_ROLLING_YEAR=False → constant 365 days always
    """
    days_in_period = (period_end - period_start).days
    if days_in_period <= 0:
        return 0.0

    if ABLATION_ROLLING_YEAR:
        rolling_year_end = period_start + relativedelta(years=1)
        days_in_year = (rolling_year_end - period_start).days
    else:
        days_in_year = 365  # ablation: constant, no leap-year awareness

    return P * (r_percent / 100.0) * (days_in_period / days_in_year)

In [85]:
from dateutil.relativedelta import relativedelta

def rolling_year_days(start_date):
    """
    Returns exact rolling 12-month days from start_date to same date next year.
    Handles leap years correctly.
    """
    if ABLATION_ROLLING_YEAR:
        return (start_date + relativedelta(years=1) - start_date).days
    else:
        return 365  # ablation: constant year, no leap-year awareness

In [86]:
def agent3_calculate(agent1_json, agent2_plan):
    """
    Calculates FD interest.

    KEY FIX: Cumulative FD exponent (4*T) is now computed from tenure_value+tenure_unit
    directly — NOT from tenure_days/365 which causes floating point drift.

    Examples of old error:
      5 years spanning leap year → tenure_days=1826 → T=1826/365=5.0027 → 4T=20.011 → WRONG
      18 months → tenure_days=547 → T=547/365=1.4986 → 4T=5.9945 → WRONG

    New approach:
      5 years  → 4*5  = 20 exact quarters
      18 months→ 18/3 = 6  exact quarters
      4 quarters→ 4   = 4  exact quarters
    """
    P           = agent1_json["principal"]
    r           = agent1_json["interest_rate_annual_percent"]
    T           = agent1_json["tenure_years"]          # kept for periodic_payout schedule
    start_date  = agent1_json["start_date"]
    tenure_days = agent1_json.get("tenure_days")

    # ── Exact compounding periods for cumulative FD ────────────────────────────
    # Use tenure_value + tenure_unit from LLM (stored by parse_tenure_from_llm)
    # to get exact integer/rational quarter count — zero floating point drift.
    tenure_value = agent1_json.get("tenure_value")
    tenure_unit  = str(agent1_json.get("tenure_unit", "years")).lower().strip()

    def get_exact_quarters(tv, tu):
        """Return exact number of quarterly compounding periods."""
        tv = float(tv)
        if tu == "years":
            return 4 * int(tv)          # 5y → 20, 2y → 8  (always integer)
        elif tu == "months":
            return tv / 3               # 18m → 6.0, 9m → 3.0
        elif tu == "quarters":
            return float(tv)            # 4Q → 4.0
        elif tu == "days":
            return 4 * tv / 365.25      # best approximation for day-based tenures
        else:
            return 4 * T                # fallback to old tenure_years method

    if tenure_value is not None and ABLATION_EXACT_N:
        n_quarters = get_exact_quarters(tenure_value, tenure_unit)
    else:
        # Ablation: approximate n from tenure_days (floating point drift)
        tenure_years_approx = agent1_json.get("tenure_years", agent1_json.get("tenure_days", 365) / 365)
        n_quarters = tenure_years_approx * 4

    calc_mode              = agent2_plan["calc_mode"]
    periodicity            = agent2_plan["periodicity"]
    question_scope         = agent2_plan["question_scope"]
    requested_period_index = agent2_plan.get("requested_period_index")

    results = {
        "per_period_interest": [],
        "total_interest":    0.0,
        "total_net_no_tds":  0.0,
        "total_net_tds_10":  0.0,
        "total_net_tds_20":  0.0,
        "maturity_amount":   None,
        "question_scope":    question_scope,
    }

    # ── CUMULATIVE FD ──────────────────────────────────────────────────────────
    if calc_mode == "cumulative":
        maturity = P * (1 + r / 400.0) ** n_quarters
        interest = maturity - P

        # For display: show clean integer T when possible
        T_display = int(n_quarters // 4) if n_quarters == int(n_quarters) and int(n_quarters) % 4 == 0 else round(n_quarters / 4, 4)

        results["total_interest"]   = round(interest, 2)
        results["total_net_no_tds"] = round(interest, 2)
        results["total_net_tds_10"] = round(interest * 0.90, 2)
        results["total_net_tds_20"] = round(interest * 0.80, 2)
        results["maturity_amount"]  = round(maturity, 2)
        results["formula"] = (
            f"A = P x (1 + r/400)^(4xT)  |  "
            f"P={P:,.0f}, r={r}%, T={T_display}  |  "
            f"Maturity = Rs. {maturity:,.2f}"
        )

    # ── PERIODIC PAYOUT (monthly / quarterly / half-yearly) ───────────────────
    # Simple interest: Interest = P x (r/100) x (days / rolling_year_days)
    elif calc_mode == "periodic_payout":
        periods = build_schedule(start_date, T, periodicity, None, tenure_days, True)
        first_rolling_year = None

        for i, (p_start, p_end) in enumerate(periods, start=1):
            interest = interest_for_period(P, r, p_start, p_end)
            ryd = rolling_year_days(p_start)

            if first_rolling_year is None:
                first_rolling_year = ryd

            results["per_period_interest"].append({
                "period":            i,
                "start":             p_start.strftime("%Y-%m-%d"),
                "end":               p_end.strftime("%Y-%m-%d"),
                "days":              (p_end - p_start).days,
                "rolling_year_days": ryd,
                "interest":          round(interest, 2),
                "net_no_tds":        round(interest, 2),
                "net_tds_10":        round(interest * 0.90, 2),
                "net_tds_20":        round(interest * 0.80, 2),
            })
            results["total_interest"] += interest

        results["total_interest"]   = round(results["total_interest"], 2)
        results["total_net_no_tds"] = round(results["total_interest"], 2)
        results["total_net_tds_10"] = round(results["total_interest"] * 0.90, 2)
        results["total_net_tds_20"] = round(results["total_interest"] * 0.80, 2)
        results["formula"] = (
            f"Interest = P x r/100 x days/{first_rolling_year}  "
            f"(P={P:.0f}, r={r}%)"
        )

    return results

In [87]:
# =========================================================
# 🔹 SIMPLIFIED format_response_for_user
# This function now just passes data through to the explainer
# =========================================================

def format_response_for_user(calculation_results, agent2_plan, user_query):
    """
    Simplified function that just returns the calculation results.
    The actual formatting is handled by run_agent_explainer.
    """
    # Just return the calculation results as-is
    # The LLM explainer will handle all formatting
    return calculation_results


In [88]:
# =========================================================
# FINAL agent4_premature FUNCTION
# =========================================================

def agent4_premature(agent1_json, agent2_plan):
    """
    Premature FD calculator with detailed calculation steps.
    """

    P = agent1_json["principal"]
    start_date_str = agent1_json["start_date"]
    withdrawal_date_str = agent1_json["withdrawal_date"]
    holding_days = agent1_json["holding_days"]

    payout_type = agent1_json["payout_type"]
    contracted_rate = agent1_json.get("interest_rate_annual_percent")
    effective_rate = agent1_json.get("effective_rate_for_holding_period")
    penalty_rate = 0.5

    if effective_rate is None:
        return {
            "error": "rate_not_found",
            "message": "Effective rate for holding period not found.",
            "agent1_json": agent1_json
        }

    start_date = datetime.strptime(start_date_str, "%Y-%m-%d").date()
    withdrawal_date = datetime.strptime(withdrawal_date_str, "%Y-%m-%d").date()

    applicable_rate = max(min(effective_rate,contracted_rate)- penalty_rate, 0.0)
    rate_basis = "effective_only"

    den_days = rolling_year_days(start_date) if ABLATION_ROLLING_YEAR else 365

    results = {
        "type": "premature_fd",
        "payout_type": payout_type,
        "start_date": start_date_str,
        "withdrawal_date": withdrawal_date_str,
        "holding_days": holding_days,
        "contracted_rate": contracted_rate,
        "effective_rate": effective_rate,
        "penalty_rate": penalty_rate,
        "applicable_rate_after_penalty": applicable_rate,
        "rate_basis": rate_basis,
        "rolling_year_days": den_days,
        "formulas_used": [],
        "calculation_steps": []
    }

    results["formulas_used"].append({
        "formula_name": "Premature Withdrawal Penalty",
        "penalty_rule": "Applicable Rate = Effective Rate - 0.5%",
        "explanation": {
            "effective_rate_for_holding_period": f"{effective_rate}%",
            "penalty_deduction": f"{penalty_rate}%",
            "final_applicable_rate": f"{applicable_rate}%"
        }
    })

    results["calculation_steps"].append({
        "step": 1,
        "description": "Determine applicable interest rate after penalty",
        "calculation": f"{effective_rate}% - {penalty_rate}%",
        "result": f"{applicable_rate}%"
    })

    # CASE A – CUMULATIVE FD
    if payout_type == "cumulative":
        T_eff_years = holding_days / den_days
        maturity = P * (1 + applicable_rate / 400.0) ** (4 * T_eff_years)
        interest = maturity - P

        results["formulas_used"].append({
            "formula_name": "Cumulative FD Premature Withdrawal",
            "formula": "A = P x (1 + r/400)^(4xT)",
            "where": {
                "P": f"Rs. {P:,.2f}",
                "r": f"{applicable_rate}% (after penalty)",
                "T": f"{T_eff_years:.4f} years ({holding_days} days)"
            }
        })

        results["calculation_steps"].append({
            "step": 2,
            "description": "Calculate tenure in years",
            "calculation": f"{holding_days} days / {den_days} days",
            "result": f"{T_eff_years:.4f} years"
        })

        results["calculation_steps"].append({
            "step": 3,
            "description": "Calculate maturity amount",
            "calculation": f"Rs. {P:,.2f} x (1 + {applicable_rate}/400)^(4x{T_eff_years:.4f})",
            "intermediate": f"Rs. {P:,.2f} x {(1 + applicable_rate/400)**(4*T_eff_years):.6f}",
            "result": f"Rs. {maturity:,.2f}"
        })

        results["calculation_steps"].append({
            "step": 4,
            "description": "Calculate interest earned",
            "calculation": f"Rs. {maturity:,.2f} - Rs. {P:,.2f}",
            "result": f"Rs. {interest:,.2f}"
        })

        results.update({
            "interest_revised": round(interest, 2),
            "already_paid_interest": 0.0,
            "net_interest_after_adjustment": round(interest, 2),
            "settlement_amount": round(maturity, 2),
            "mode": "cumulative_true_compounding"
        })
        return results

    # CASE B – PAYOUT FD
    periodicity = agent2_plan.get("periodicity")
    if not periodicity or periodicity == "on_maturity":
        if payout_type == "quarterly_payout":
            periodicity = "quarterly"
        elif payout_type == "monthly_payout":
            periodicity = "monthly"
        elif payout_type == "half_yearly_payout":
            periodicity = "half_yearly"
        else:
            periodicity = "quarterly"

    penal_interest = P * (applicable_rate / 100.0) * (holding_days / den_days)

    results["formulas_used"].append({
        "formula_name": "Payout FD Premature Withdrawal",
        "formula": "Interest = P x (r/100) x (holding_days / rolling_year_days)",
        "where": {
            "P": f"Rs. {P:,.2f}",
            "r": f"{applicable_rate}% (after penalty)",
            "holding_days": f"{holding_days}",
            "rolling_year_days": f"{den_days}"
        },
        "adjustment": "Subtract already paid interest for completed periods"
    })

    results["calculation_steps"].append({
        "step": 2,
        "description": "Calculate total interest at penal rate",
        "calculation": f"Rs. {P:,.2f} x ({applicable_rate}/100) x ({holding_days}/{den_days})",
        "result": f"Rs. {penal_interest:,.2f}"
    })

    already_paid = 0.0
    period_details = []

    if contracted_rate is not None:
        periods = build_schedule(
            start_date_str=start_date_str,
            tenure_years=None,
            periodicity=periodicity,
            until_date_str=withdrawal_date_str,
            tenure_days=None,
            allow_partial_final=False
        )

        for i, (p_start, p_end) in enumerate(periods, start=1):

            int_paid = interest_for_period(P, contracted_rate, p_start, p_end)
            int_penal = interest_for_period(P, applicable_rate, p_start, p_end)

            if p_end <= withdrawal_date:
                already_paid += int_paid

            ryd = rolling_year_days(p_start)
            period_details.append({
                "period_index": i,
                "start": p_start.strftime("%Y-%m-%d"),
                "end": p_end.strftime("%Y-%m-%d"),
                "days": (p_end - p_start).days,
                "rolling_year_days": ryd,
                "interest_paid_at_contracted_rate": round(int_paid, 2),
                "interest_at_penal_rate_for_period": round(int_penal, 2)
            })

        results["calculation_steps"].append({
            "step": 3,
            "description": "Calculate already paid interest (completed periods only)",
            "period_breakdown": period_details,
            "total_already_paid": f"Rs. {already_paid:,.2f}"
        })

    net_interest = penal_interest - already_paid
    settlement_amount = P + net_interest

    results["calculation_steps"].append({
        "step": 4,
        "description": "Calculate net interest after adjustment",
        "calculation": f"Rs. {penal_interest:,.2f} - Rs. {already_paid:,.2f}",
        "result": f"Rs. {net_interest:,.2f}"
    })

    results["calculation_steps"].append({
        "step": 5,
        "description": "Calculate settlement amount",
        "calculation": f"Rs. {P:,.2f} + Rs. {net_interest:,.2f}",
        "result": f"Rs. {settlement_amount:,.2f}"
    })

    results.update({
        "interest_should_be_at_penal_rate": round(penal_interest, 2),
        "already_paid_interest": round(already_paid, 2),
        "net_interest_after_adjustment": round(net_interest, 2),
        "settlement_amount": round(settlement_amount, 2),
        "period_details": period_details,
        "mode": "payout_simple_interest_with_adjustment"
    })

    return results


In [89]:
agent_explainer_prompt = """You are the final response formatter for FD (Fixed Deposit) queries.

Your job is to take calculation results and format them EXACTLY like the golden answer examples below.

===================================================================
CRITICAL RULES:
===================================================================
1. Match the EXACT format and structure of golden answers
2. Use "Rs." for currency (NOT rupee symbol)
3. Use "x" for multiplication (be consistent with examples)
4. Show complete calculations with all intermediate steps
5. Use the exact date format from examples (e.g., "5 Jan 2025")
6. Match the level of detail - not more, not less
7. Use similar phrasing and tone as golden answers
8. If there is a mismatch in provided interest rate as compared to actual, flag it. However do the entire calculation on correct interest rate and show the answer
9. Rolling year days: NEVER hardcode 365. Each period has its own "rolling_year" value.
   - Use rolling_year from each period's data in the calculation shown.
   - Period starting in a leap year window (e.g., Feb 2024) -> rolling_year = 366
   - Period starting outside leap year window -> rolling_year = 365
   - Show per-period rolling year ONLY when they differ across periods.
   - When showing a single period (Q1 query), show "Rolling year days: {that period's rolling_year}"
===================================================================
MATH FORMATTING RULES (CRITICAL - READ BEFORE ANSWERING):
===================================================================
- Write ALL formulas in plain text only. NO LaTeX of any kind.
- NO \( \) or \[ \] or \frac or \dfrac or \times or \mathbf or \left or \right
- NO superscript notation like ^{20} — write as ^20 instead
- NO markdown bold (**text**) for numbers or formulas

CORRECT formula style (always use this):
  A = P x (1 + r/400)^(4 x T)
  A = 2,00,000 x (1 + 6.35/400)^(4 x 5)
  A = 2,00,000 x (1.015875)^20
  A = 2,00,000 x 1.3702678
  A = Rs. 2,74,053.56

WRONG formula style (NEVER use):
  A = P \times \left(1 + \dfrac{r}{400}\right)^{4 \times T}
  \mathbf{Rs.\ 2,74,053.56}
  **Rs. 2,74,053.56**

Division must be written as: r/400   NOT  \dfrac{r}{400}
Powers must be written as:   (1.015875)^20   NOT  (1.015875)^{20}
Currency must be written as: Rs. 2,74,053.56   NOT  \mathbf{Rs.\ 2,74,053.56}

===================================================================
GOLDEN ANSWER EXAMPLES (Learn from these):
===================================================================

EXAMPLE 1: Single Quarter Query
-------------------------------------------------------------------
Query: "I booked an FD of Rs. 2,50,000 on 5th January 2025 for 18 months. I am a normal citizen. How much 1st quarterly interest will I receive?"

Golden Answer:
FD booking period: "Between Nov 12, 2024 - Jun 14, 2025"
# ↑ This comes from "Rate Source" in FD DETAILS. Format: "Rate card: 12-Nov-2024 to 14-Jun-2025" → "Between Nov 12, 2024 - Jun 14, 2025"

Tenure: 1 to <2 years -> 6.75% (regular citizen)

Quarterly payout FD -> exact days per quarter

Rolling year days: {rolling_year_days from calculation results}

Interest = Principal x Rate x (Days / Year)
Interest = 2,50,000 x 0.0675 x (90 / 365)
Interest = Rs. 4,160.96

Approx. Rs. 4,160.96 credited at end of first quarter

-------------------------------------------------------------------

EXAMPLE 2: Full Breakdown Query
-------------------------------------------------------------------
Query: "FD booked Rs. 50,000 on 20 June 2025 for 400 days, senior citizen. Interest payout quarterly. How much will I get each quarter?"

Golden Answer:
Principal (P): Rs. 50,000
Start date: 20 June 2025
Maturity: 400 days later -> 25 July 2026
Senior citizen rate: 7.10% p.a.
Type: Quarterly payout FD

Interest calculation: Exact day count per quarter
Rolling year = exact 365/366-day rolling year from each quarter start (use EXACT value from calculation results, NOT always 365)

Period                          Days  Calculation                          Interest
20 Jun 2025 -> 20 Sep 2025       92   50,000 x 0.071 x (92/365)           Rs.   894.79
20 Sep 2025 -> 20 Dec 2025       91   50,000 x 0.071 x (91/365)           Rs.   885.07
20 Dec 2025 -> 20 Mar 2026       90   50,000 x 0.071 x (90/365)           Rs.   875.34
20 Mar 2026 -> 20 Jun 2026       92   50,000 x 0.071 x (92/365)           Rs.   894.79
20 Jun 2026 -> 25 Jul 2026       35   50,000 x 0.071 x (35/365)           Rs.   340.41
(final partial)

Final Accurate Payout Summary
-----------------------------------------------------
Component                                    Amount
-----------------------------------------------------
Total quarterly payouts credited             Rs. 3,549.99
Final partial payout at maturity (35 days)   Rs.   340.41
-----------------------------------------------------
Total Interest on maturity                   Rs. 3,890.40
-----------------------------------------------------

You will receive 4 quarterly credits during the FD period, followed by a final partial interest payout at maturity.
Your total earnings over the 400-day tenure will be Rs. 3,890.40.

-------------------------------------------------------------------

EXAMPLE 3: Cumulative FD Query
-------------------------------------------------------------------
Query: "I deposited Rs. 1,00,000 for 2 years in cumulative FD at 7% p.a. What will I get at maturity?"

Golden Answer:
FD Type: Cumulative with Quarterly Compounding
Principal: Rs. 1,00,000
Rate: 7.00% p.a. (regular citizen)
Tenure: 2 years

Formula: A = P x (1 + r/400)^(4 x T)

Calculation:
A = 1,00,000 x (1 + 7/400)^(4 x 2)
A = 1,00,000 x (1.0175)^8
A = 1,00,000 x 1.14868
A = Rs. 1,14,868

Total Interest = Rs. 1,14,868 - Rs. 1,00,000 = Rs. 14,868

At maturity, you will receive Rs. 1,14,868 (earning Rs. 14,868 in interest).

-------------------------------------------------------------------

EXAMPLE 4: Cumulative FD - 5 Year
-------------------------------------------------------------------
Query: "FD Rs. 2,00,000 on 5 June 2025 for 5 years, regular citizen, cumulative quarterly. What is maturity amount?"

Golden Answer:
FD Type: Cumulative with Quarterly Compounding
Principal: Rs. 2,00,000
Rate: 6.35% p.a. (regular citizen)
Tenure: 5 years

Formula: A = P x (1 + r/400)^(4 x T)

Calculation:
Step 1: r/400 = 6.35/400 = 0.015875
Step 2: 1 + 0.015875 = 1.015875
Step 3: 4 x T = 4 x 5 = 20
Step 4: (1.015875)^20 = 1.3702678
Step 5: A = 2,00,000 x 1.3702678 = Rs. 2,74,053.56

Total Interest = Rs. 2,74,053.56 - Rs. 2,00,000 = Rs. 74,053.56

At maturity, you will receive Rs. 2,74,053.56 (earning Rs. 74,053.56 in interest).

-------------------------------------------------------------------

===================================================================
KEY FORMATTING PATTERNS TO FOLLOW:
===================================================================

1. For Specific Period Queries (Q1, Q2, etc.):
   - Start with booking period and tenure
     * "FD booking period" dates come from the "Rate Source" field in FD DETAILS
     * Rate Source format: "Rate card: DD-Mon-YYYY to DD-Mon-YYYY"
     * Convert to: "Between Mon DD, YYYY - Mon DD, YYYY"
     * Example: "Rate card: 12-Nov-2024 to 14-Jun-2025" → "Between Nov 12, 2024 - Jun 14, 2025"
     * NEVER copy dates from the examples above — always read from the current FD DETAILS
   - Show payout type description
   - Show period dates with exact days
   - Show rolling year days
   - Show formula with substituted values
   - End with final amount credited

2. For Full Breakdown Queries (all quarters):
   - Start with Principal, Start date, Maturity
   - Show rate and type
   - Create a plain-text table with columns: Period | Days | Calculation | Interest
   - Show final summary table with totals
   - End with total earnings summary sentence

3. For Cumulative FDs:
   - Show FD type, principal, rate, tenure
   - Show formula in plain text (NO LaTeX)
   - Show step-by-step calculation (Step 1, Step 2, ...)
   - Show total interest and maturity amount

4. Date Formatting:
   - Use: "5 Jan 2025" or "20 June 2025" (NOT "2025-01-05")
   - Use: "5 Jan 2025 -> 4 Apr 2025" for period ranges (use -> NOT arrow symbols)

5. Number Formatting:
   - Use: "Rs. 4,160.96" (with comma for thousands)
   - Use: "2,50,000" or "50,000" (Indian numbering style)
   - Use: "0.0675" for rate decimals in formulas
   - Use: "6.75%" for percentage display in text

6. Calculation Display:
   - Always show: Principal x Rate x (Days/Year)
   - Always substitute actual values
   - Use = for exact, use approx. for approximate results

===================================================================
YOUR TASK:
===================================================================
You will receive:
1. User's question
2. FD details from Agent 1 (principal, dates, rates, tenure)
3. Calculation plan from Agent 2
4. Detailed calculation results from Agent 3/4

Generate a response that:
- Matches the EXACT style of golden answers above
- Uses the calculation data provided
- Follows the formatting patterns
- Is clear and user-friendly

DO NOT:
- Add information not in the calculation results
- Use different formatting than examples
- Skip calculation steps shown in examples
- Use rupee symbol - use "Rs." instead
- Use LaTeX of any kind (\(, \), \[, \], \frac, \dfrac, \times, \mathbf, \left, \right)
- Use markdown bold (**) for numbers or results
- Use superscript notation ^{n} - write as ^n instead

IMPORTANT:
- For specific period queries (Q1, Q2), show ONLY that period's calculation (like Example 1)
- For full breakdown queries, show ALL periods in a table (like Example 2)
- Match the tone and completeness of golden answers
- Principal does NOT get paid at every interest payout cycle. Principal comes at the end of tenure only.

===================================================================
SPECIAL INSTRUCTION FOR TDS QUERIES:
===================================================================
When user mentions "received only", "got only", or states a specific amount received:
1. Check if amount matches: gross interest - 10% TDS or gross interest - 20% TDS
2. Explain TDS was deducted
3. Show: Gross -> TDS -> Net calculation
4. Mention Form 15H/15G eligibility
5. Use EXACT calculated data from Agent 3 (do not recalculate days or interest)
"""


<>:26: SyntaxWarning: invalid escape sequence '\('
<>:26: SyntaxWarning: invalid escape sequence '\('
/tmp/ipykernel_2705/2587792016.py:26: SyntaxWarning: invalid escape sequence '\('
  - NO \( \) or \[ \] or \frac or \dfrac or \times or \mathbf or \left or \right


In [90]:
def run_agent_explainer(user_query, agent1_json, agent2_plan, calculation_results):
    """
    Final response formatter.
    FIX 1: Pre-computed cumulative steps — LLM never re-derives math.
    FIX 2: Indian number formatting (indian_format) used everywhere in
            cumulative_steps_block so LLM receives correct commas and
            does NOT attempt to reformat large amounts incorrectly.
    """

    # ── Indian number formatter ────────────────────────────────────────────────
    def indian_format(n, decimals=2):
        """Format number in Indian numbering (e.g. 58,35,362.50).
        Whole numbers (no fractional part) are shown without decimals."""
        n = round(float(n), decimals)
        is_whole = (n == int(n))
        if is_whole:
            integer_part = str(int(n))
            decimal_part = None
        else:
            s = f'{n:.{decimals}f}'
            integer_part, decimal_part = s.split('.')

        if len(integer_part) <= 3:
            result = integer_part
        else:
            last3 = integer_part[-3:]
            rest  = integer_part[:-3]
            groups = []
            while len(rest) > 2:
                groups.append(rest[-2:])
                rest = rest[:-2]
            if rest:
                groups.append(rest)
            groups.reverse()
            result = ','.join(groups) + ',' + last3

        return f'{result}.{decimal_part}' if decimal_part is not None else result

    # ── Common values ──────────────────────────────────────────────────────────
    calc_mode    = agent2_plan.get("calc_mode")
    periods      = calculation_results.get("per_period_interest")
    total_int    = calculation_results.get("total_interest", 0)
    maturity_amt = calculation_results.get("maturity_amount")
    formula      = calculation_results.get("formula")
    is_premature = (agent1_json.get("is_premature", False)
                    or calculation_results.get("type") == "premature_fd")

    # ── PREMATURE WITHDRAWAL ───────────────────────────────────────────────────
    if is_premature:
        calc_steps = calculation_results.get("calculation_steps", [])
        formulas   = calculation_results.get("formulas_used", [])

        formula_display = ""
        if formulas:
            f = formulas[1] if len(formulas) > 1 else formulas[0]
            where_text = ", ".join(f"{k} = {v}" for k, v in f.get("where", {}).items())
            formula_display = f"Formula: {f.get('formula')}  |  {where_text}"

        steps_text = []
        for step in calc_steps:
            s = f"Step {step.get('step')}: {step.get('description')}\n  Calc: {step.get('calculation')}"
            if step.get("intermediate"):
                s += f"\n  Intermediate: {step.get('intermediate')}"
            s += f"\n  Result: {step.get('result')}"
            steps_text.append(s)

        prompt = f"""USER QUESTION: {user_query}

FD BOOKING DETAILS:
- Principal: Rs. {indian_format(agent1_json['principal'])}
- Booking Date: {agent1_json['start_date']}
- Contracted Tenure: {agent1_json['tenure_years']} years ({agent1_json['tenure_days']} days)
- Contracted Rate: {agent1_json['interest_rate_annual_percent']}%
- Payout Type: {calculation_results.get('payout_type', 'cumulative')}
- Citizenship: {agent1_json['citizenship_status']}

PREMATURE WITHDRAWAL DETAILS:
- Withdrawal Date: {agent1_json.get('withdrawal_date', 'NA')}
- Holding Days: {agent1_json.get('holding_days', 'NA')}
- Effective Rate for Holding Period: {calculation_results.get('effective_rate', 'NA')}%
- Penalty Rate: {calculation_results.get('penalty_rate', 0.5)}%
- Applicable Rate After Penalty: {calculation_results.get('applicable_rate_after_penalty', 'NA')}%
- Rolling Year Days: {calculation_results.get('rolling_year_days', 365)}

{formula_display}

CALCULATION STEPS:
{chr(10).join(steps_text)}

FINAL RESULTS:
- Interest Earned: Rs. {indian_format(calculation_results.get('net_interest_after_adjustment', 0))}
- Settlement Amount: Rs. {indian_format(calculation_results.get('settlement_amount', 0))}
- Already Paid Interest: Rs. {indian_format(calculation_results.get('already_paid_interest', 0))}

FORMAT: Show step-by-step premature withdrawal calculation. Use Rs. not symbol, use x for multiplication."""

    # ── NORMAL FD ──────────────────────────────────────────────────────────────
    else:
        # ── Pre-compute cumulative steps in Python ─────────────────────────────
        cumulative_steps_block = ""
        if calc_mode == "cumulative" and maturity_amt:
            P  = agent1_json["principal"]
            r  = agent1_json["interest_rate_annual_percent"]
            tv = agent1_json.get("tenure_value")
            tu = str(agent1_json.get("tenure_unit", "years")).lower()

            # Exact quarters — same logic as agent3_calculate
            if tv is not None:
                tv_f = float(tv)
                if tu == "years":
                    n         = 4 * int(tv_f)
                    T_display = int(tv_f)
                elif tu == "months":
                    n         = tv_f / 3
                    T_display = f"{tv_f} months"
                elif tu == "quarters":
                    n         = float(tv_f)
                    T_display = f"{int(tv_f)} quarters"
                else:
                    n         = 4 * agent1_json.get("tenure_years", 1)
                    T_display = agent1_json.get("tenure_years", 1)
            else:
                n         = 4 * agent1_json.get("tenure_years", 1)
                T_display = agent1_json.get("tenure_years", 1)

            step1_val         = round(r / 400, 10)
            step2_val         = round(1 + step1_val, 10)
            step3_val         = n
            step4_val         = round(step2_val ** step3_val, 7)
            maturity_computed = round(P * step4_val, 2)
            interest_computed = round(maturity_computed - P, 2)

            # Use indian_format for ALL amounts so LLM never reformats them
            cumulative_steps_block = f"""
CUMULATIVE FD — PRE-COMPUTED STEPS (copy EXACTLY, do NOT recalculate):
  P = Rs. {indian_format(P)}
  r = {r}%
  T = {T_display}
  n (compounding periods) = {step3_val}

  Step 1: r/400 = {r}/400 = {step1_val}
  Step 2: 1 + {step1_val} = {step2_val}
  Step 3: 4 x T = {step3_val}
  Step 4: ({step2_val})^{step3_val} = {step4_val}
  Step 5: A = {indian_format(P)} x {step4_val} = Rs. {indian_format(maturity_computed)}
  Total Interest = Rs. {indian_format(maturity_computed)} - Rs. {indian_format(P)} = Rs. {indian_format(interest_computed)}

STRICT RULES:
- Copy ALL values above EXACTLY as shown — do NOT recalculate or reformat.
- Numbers already use Indian comma format (e.g. 58,35,362.50). Keep them as-is.
- Do NOT show an alternate Step 5.
- Do NOT show (1 + r/4/100) style — it is identical to r/400, no separate line needed.
- Do NOT use = prefix (Excel style). Use: {indian_format(P)} x {step4_val} = Rs. {indian_format(maturity_computed)}
"""

        # Build period table
        period_table = ""
        if periods:
            rows = [
                f"Period {p['period']}: {p['start']} to {p['end']} "
                f"({p['days']} days, rolling_year={p.get('rolling_year_days', 365)}) | "
                f"Gross: Rs. {indian_format(p['interest'])} | "
                f"Net(0% TDS): Rs. {indian_format(p['net_no_tds'])} | "
                f"Net(10% TDS): Rs. {indian_format(p['net_tds_10'])} | "
                f"Net(20% TDS): Rs. {indian_format(p['net_tds_20'])}"
                for p in periods
            ]
            period_table = "PERIOD SCHEDULE (all TDS scenarios):\n" + "\n".join(rows)

        tds_summary = f"""
TDS SUMMARY (for LLM reference — do NOT show by default):
- Gross Total Interest:     Rs. {indian_format(total_int)}
- Net Total (0%  / no TDS): Rs. {indian_format(calculation_results.get('total_net_no_tds', total_int))}
- Net Total (10% TDS):      Rs. {indian_format(calculation_results.get('total_net_tds_10', 0))}
- Net Total (20% TDS):      Rs. {indian_format(calculation_results.get('total_net_tds_20', 0))}
"""

        prompt = f"""Today's date is {TODAY_STR}. USER QUESTION: {user_query}

FD DETAILS:
- Principal: Rs. {indian_format(agent1_json['principal'])}
- Rate: {agent1_json['interest_rate_annual_percent']}% p.a.
- Tenure: {agent1_json.get('tenure_value')} {agent1_json.get('tenure_unit')} ({agent1_json.get('tenure_days')} days)
- Booking Date: {agent1_json['start_date']}
- Maturity Date: {agent1_json.get('maturity_date', 'N/A')}
- Type: {agent1_json['payout_type']}
- Citizenship: {agent1_json['citizenship_status']} citizen
- Rate Source: {agent1_json['rate_source']}

CALCULATION PLAN:
- Calc Mode: {calc_mode}
- Periodicity: {agent2_plan.get('periodicity')}
- Question Scope: {agent2_plan.get('question_scope')}
- Requested Period Index: {agent2_plan.get('requested_period_index')}

{cumulative_steps_block}
{period_table}
{tds_summary}

GROSS TOTAL INTEREST: Rs. {indian_format(total_int)}

TDS DISPLAY RULES (STRICT):
1. DEFAULT: Show ONLY gross interest. Do NOT show any TDS column or TDS summary row.

2. IF user mentions a specific amount received ("received only X", "got only X",
   "why less", "I got X"):

   STEP A — Show gross calculation first (exactly as normal).

   STEP B — Show all 4 TDS scenarios:
   Gross Interest:          Rs. [gross]
   After 0%  TDS (Form 15H/15G): Rs. [gross x 1.00]
   After 5%  TDS:           Rs. [gross x 0.95]
   After 10% TDS (PAN):     Rs. [gross x 0.90]
   After 20% TDS (no PAN):  Rs. [gross x 0.80]

   STEP C — Find closest match:
   Compute delta = abs(each scenario - received_amount).
   Pick scenario with smallest delta.

   STEP D — If smallest delta <= Rs. 10:
   Output exactly this:
   "✅ Your received amount of Rs. [received] matches the [X%] TDS scenario.
   Gross Rs. [gross] → TDS Rs. [tds_deducted] → Net Rs. [received].
   [If 0%: Your Form 15H/15G is working correctly, no TDS was deducted.]
   [If 10%: TDS was deducted as PAN is registered. Submit Form 15H (senior)
    or 15G (regular) to avoid future deductions if your income is within limits.]
   [If 20%: TDS was deducted at 20% as PAN may not be linked. Please link your
    PAN with the bank to reduce TDS to 10%.]"

   STEP E — If smallest delta > Rs. 10 (NO scenario matches):
   Output exactly this:
   "⚠️ Your received amount of Rs. [received] does not match any standard TDS rate.

   Here is what you should have received under each scenario:
   Gross Interest:    Rs. [gross]
   0%  TDS:           Rs. [gross x 1.00]
   5%  TDS:           Rs. [gross x 0.95]
   10% TDS:           Rs. [gross x 0.90]
   20% TDS:           Rs. [gross x 0.80]

   The most likely reason is annual TDS shortfall adjustment.
   Banks calculate TDS on your TOTAL interest across ALL FDs held with them
   for the full financial year. If TDS was under-deducted in earlier quarters,
   the bank recovers the shortfall in a later quarter — which can make the
   deduction appear unusually high for that specific payout.

   We recommend checking your Form 26AS on the Income Tax portal to see the
   exact TDS amount deposited by your bank against your PAN. You can also
   contact your bank's branch and request a TDS computation statement for
   clarity."

3. NEVER show TDS scenarios unless triggered by Step 2 above.
4. NEVER show Form 15H/15G advice unless it is directly relevant per Step D above.
RATE MISMATCH CHECK (MANDATORY — check this for EVERY query):
- If the user mentioned a specific interest rate in their question, compare it to the
  actual rate from FD DETAILS.
- If they differ, you MUST start your answer with:

  ⚠️ Rate Correction: You mentioned [X]% but the applicable rate for your FD
  ([tenure], [citizen type], booked [date]) is [actual]% p.a. as per the rate card.
  All calculations below use [actual]%.

- Then proceed with the full calculation using the correct rate.
- If the user did NOT mention a rate, skip this check entirely.
FUTURE BOOKING DATE CHECK (MANDATORY):
- Compare the Booking Date from FD DETAILS against today's date ({TODAY_STR}).
- If Booking Date > today:
  Perform the full calculation normally using the current rate card.
  Then add this note at the END of your answer (after all calculations):

  "📌 Note: This calculation is based on the current interest rate of [X]% p.a.
  Since your FD booking date ([date]) is in the future, the applicable rate may
  change before then. Please recheck the rate card at the time of actual booking."

- If Booking Date <= today: skip this check entirely, show no note.

Generate a complete answer matching the golden answer format."""

    # ── LLM CALL ───────────────────────────────────────────────────────────────
    num_periods = len(periods) if periods else 0
    max_tokens  = max(2000, num_periods * 120 + 800)

    try:
        resp = client.chat.completions.create(
            model=get_model("agent_explainer"),
            messages=[
                {"role": "system", "content": agent_explainer_prompt},
                {"role": "user",   "content": prompt}
            ],
            temperature=0.1,
            max_tokens=max_tokens,
        ).choices[0].message.content

        cleaned = clean_unicode_text(resp.strip())

        if not cleaned or len(cleaned) < 20:
            if is_premature:
                return (
                    f"Premature withdrawal after {agent1_json.get('holding_days')} days.\n"
                    f"Settlement Amount: Rs. {indian_format(calculation_results.get('settlement_amount', 0))}\n"
                    f"Interest Earned: Rs. {indian_format(calculation_results.get('net_interest_after_adjustment', 0))}"
                )
            elif maturity_amt:
                return f"Maturity Amount: Rs. {indian_format(maturity_amt)}  |  Interest: Rs. {indian_format(total_int)}"
            elif periods:
                lines = [f"{'Q':<4} {'Start':<14} {'End':<14} {'Days':>6} {'Interest':>14}", "-" * 56]
                for p in periods:
                    lines.append(f"Q{p['period']:<3} {p['start']:<14} {p['end']:<14} {p['days']:>6} Rs. {p['interest']:>11,.2f}")
                lines.append(f"\nTotal Interest: Rs. {indian_format(total_int)}")
                return "\n".join(lines)
            return f"Total interest earned: Rs. {indian_format(total_int)}"

        return cleaned

    except Exception as e:
        print(f"Agent Explainer Error: {e}")
        return f"Calculation complete. Total interest: Rs. {indian_format(total_int)}"

In [91]:
def format_response_for_user(calculation_results, agent2_plan, user_query):
    """
    Simplified function that just returns the calculation results.
    The actual formatting is handled by run_agent_explainer.
    """
    # Just return the calculation results as-is
    # The LLM explainer will handle all formatting
    return calculation_results


In [92]:
def full_router_pipeline(user_query, show_agent_outputs=False):
    """
    Enhanced pipeline with LLM-driven response formatting.

    Args:
        user_query: User's question
        show_agent_outputs: If True, returns all intermediate agent outputs

    Returns:
        dict with final_answer and optionally agent_outputs
    """

    # ========================================================
    # STEP 1: Router decides RAG or Calculation
    # ========================================================
    try:
        route_json = run_router(user_query)
        if show_agent_outputs:
            # print("=" * 80)
            # print("ROUTER OUTPUT:")
            # print(json.dumps(route_json, indent=2))
            # print("=" * 80)
            pass
    except Exception as e:
        return {
            "route": "router_error",
            "error_type": "router_failed",
            "final_answer": "I encountered an error while processing your query. Please try rephrasing your question.",
            "error_details": str(e) if show_agent_outputs else None
        }

    # ========================================================
    # CASE 1: Pure RAG policy answer
    # ========================================================
    if route_json["route"] == "rag_rules":
        try:
            answer = run_rag_rules(user_query)
            return {
                "route": "rag_rules",
                "final_answer": answer
            }
        except Exception as e:
            return {
                "route": "rag_error",
                "error_type": "rag_failed",
                "final_answer": "I couldn't retrieve the policy information. Please try again or rephrase your question.",
                "error_details": str(e) if show_agent_outputs else None
            }

    # ========================================================
    # CASE 2: Rate Query
    # ========================================================
    if route_json["route"] == "rate_query":
        try:
            result = run_agent_rate_query(user_query)
            return result
        except Exception as e:
            return {
                "route": "rate_query_error",
                "error_type": "rate_query_failed",
                "final_answer": "I couldn't retrieve the interest rate information. Please try again.",
                "error_details": str(e) if show_agent_outputs else None
            }

    # ========================================================
    # VALIDATION STEP
    # ========================================================
    if show_agent_outputs:
        print("🔍 Running validation check...")

    try:
        validation = run_validation_agent(user_query)

        if not validation["is_valid"]:
            return {
                "route": "validation_error",
                "error_type": "missing_information",
                "final_answer": validation["user_message"],
                "validation_details": {
                    "missing_fields": validation["missing_fields"],
                    "extracted_info": validation.get("extracted_info")
                } if show_agent_outputs else None
            }
    except Exception as e:
        if show_agent_outputs:
            print(f"⚠️ Validation check failed, continuing: {e}")

    # ========================================================
    # Run Agent-1
    # ========================================================
    if show_agent_outputs:
        print("🤖 Running Agent-1 (Extraction)...")

    try:
        a1 = run_agent1(user_query)
    except Exception as e:
        return {
            "route": "agent1_error",
            "error_type": "extraction_failed",
            "final_answer": "I had trouble understanding your FD details. Please provide:\n\n1. FD amount (e.g., Rs. 1,00,000)\n2. Booking date (e.g., 10-Mar-2025)\n3. Tenure (e.g., 2 years)\n\nExample: 'I booked an FD of Rs. 50,000 on 15-Jan-2025 for 18 months. How much interest will I get?'",
            "error_details": str(e) if show_agent_outputs else None
        }

    # ========================================================
    # Validate Agent-1 output
    # ========================================================
    if show_agent_outputs:
        print("✅ Validating Agent-1 output...")

    try:
        a1_validation = validate_agent1_output_with_llm(a1, user_query)

        if not a1_validation["is_valid"]:
            return {
                "route": "validation_error",
                "error_type": "invalid_extraction",
                "final_answer": a1_validation["user_message"],
                "validation_details": {
                    "missing_fields": a1_validation["missing_fields"],
                    "invalid_fields": a1_validation["invalid_fields"],
                    "extracted_data": a1
                } if show_agent_outputs else None
            }
    except Exception as e:
        if show_agent_outputs:
            print(f"⚠️ Agent1 validation failed, continuing: {e}")

    # ========================================================
    # Run Agent-2
    # ========================================================
    if show_agent_outputs:
        print("🤖 Running Agent-2 (Planning)...")

    try:
        a2 = run_agent2(a1, user_query)
    except Exception as e:
        return {
            "route": "agent2_error",
            "error_type": "planning_failed",
            "final_answer": "I had trouble understanding what calculation you need. Please try rephrasing your question.",
            "error_details": str(e) if show_agent_outputs else None
        }

    # ========================================================
    # PREMATURE WITHDRAWAL → Agent-4
    # ========================================================
    if a1.get("is_premature") is True:
        if show_agent_outputs:
            print("💰 Running Agent-4 (Premature Withdrawal)...")

        try:
            calc = agent4_premature(a1, a2)

            if calc.get("error"):
                return {
                    "route": "calculation_error",
                    "error_type": "premature_calculation_failed",
                    "final_answer": f"I couldn't calculate the premature withdrawal amount. {calc.get('message', '')}",
                    "error_details": calc if show_agent_outputs else None
                }

            final_explanation = run_agent_explainer(user_query, a1, a2, calc)

            return {
                "route": "calculation_premature",
                "final_answer": final_explanation,
                "agent_outputs": {
                    "agent1": a1,
                    "agent2": a2,
                    "calculation": calc
                } if show_agent_outputs else None
            }

        except Exception as e:
            return {
                "route": "calculation_error",
                "error_type": "premature_calculation_exception",
                "final_answer": "An error occurred while calculating premature withdrawal.",
                "error_details": str(e) if show_agent_outputs else None
            }

    # ========================================================
    # NORMAL FD → Agent-3
    # ========================================================
    else:
        if show_agent_outputs:
            print("💰 Running Agent-3 (Normal FD Calculation)...")

        try:
            calc = agent3_calculate(a1, a2)
            final_explanation = run_agent_explainer(user_query, a1, a2, calc)

            return {
                "route": "calculation_normal",
                "final_answer": final_explanation,
                "agent_outputs": {
                    "agent1": a1,
                    "agent2": a2,
                    "calculation": calc
                } if show_agent_outputs else None
            }

        except Exception as e:
            return {
                "route": "calculation_error",
                "error_type": "calculation_exception",
                "final_answer": "An error occurred during calculation.",
                "error_details": str(e) if show_agent_outputs else None
            }

In [93]:
query = "how are you? "#"Booked anfd of 1000rs at 10% interest howmuch income will i get?" #queries[0]
result = full_router_pipeline(query)
print(f"\nQ{0+1}:\n{json.dumps(result, indent=2)}")


Q1:
{
  "route": "rag_rules",
  "final_answer": "I am functioning properly, ready to provide information based on the provided context regarding FD policies and rules."
}


In [94]:
#TEST CASE 1 FOR RAG_RULES ENGINE
query = "what is the tenure for which i get the best interest rate?"#"Booked anfd of 1000rs at 10% interest howmuch income will i get?" #queries[0]
result = full_router_pipeline(query)
print(f"\nQ{0+1}:\n{json.dumps(result, indent=2)}")

📊 Found best rates for 8 categories

Q1:
{
  "route": "rate_query",
  "final_answer": "| Customer Type | NRI Status | Deposit Amount | Best Tenure | Rate |\n|---------------|------------|----------------|-------------|------|\n| Senior        | NRI        | <3CR           | 444-444 days | 7.3  |\n| Senior        | NRI        | >=3CR          | 444-444 days | 7.15 |\n| Senior        | Non NRI    | <3CR           | 444-444 days | 7.1  |\n| Senior        | Non NRI    | >=3CR          | 444-444 days | 6.85 |\n| Regular       | NRI        | <3CR           | 444-444 days | 6.8  |\n| Regular       | NRI        | >=3CR          | 444-444 days | 6.65 |\n| Regular       | Non NRI    | <3CR           | 444-444 days | 6.6  |\n| Regular       | Non NRI    | >=3CR          | 444-444 days | 6.35 |\n\nThe highest rate of 7.3% is offered to Senior NRI customers with deposits less than 3CR for a tenure of 444-444 days."
}


In [95]:
#TEST CASE 2 FOR iNTEREST RATE
query = "what is the tenure for which i get the best interest rate?"#"Booked anfd of 1000rs at 10% interest howmuch income will i get?" #queries[0]
result = full_router_pipeline(query)
print(f"\nQ{0+1}:\n{json.dumps(result, indent=2)}")

📊 Found best rates for 8 categories

Q1:
{
  "route": "rate_query",
  "final_answer": "Customer Type | NRI Status | Deposit Amount | Best Tenure | Rate |\n|---------------|------------|----------------|-------------|------|\n| Senior        | NRI        | <3CR           | 444-444 days | 7.3  |\n| Senior        | NRI        | >=3CR          | 444-444 days | 7.15 |\n| Senior        | Non NRI    | <3CR           | 444-444 days | 7  |\n| Senior        | Non NRI    | >=3CR          | 444-444 days | 6.85 |\n| Regular       | NRI <3CR           | 444-444 days | 6.8| Regular       | NRI        | >=3CR          | 444-444 days | 6.65 |\n| Regular       | Non NRI    | <3CR           | 444-444 days | 6.6  |\n| Regular       | Non NRI    | >=3CR          | 444-444 days | 6.35 |\n\nThe Senior customer category with NRI status and deposit amount less than 3CR gets the highest rate of 7.3% for a tenure of 444-444 days."
}


In [96]:
if __name__ == "__main__":
    query=queries[15] #test query for ablation run

    print("="*100)
    #print("TESTING PREMATURE WITHDRAWAL QUERY")
    print("="*100)
    print(f"Query: {query}")
    print("="*100)

    result = full_router_pipeline(query, show_agent_outputs=True)

    print("\n" + "="*100)
    print("RESULT:")
    print("="*100)
    print(json.dumps(result, indent=2, default=str))

    print("\n" + "="*100)
    print("FINAL ANSWER TO USER:")
    print("="*100)
    print(result["final_answer"])

Query: FD booked of amt 120,000 on 1jan 2023 for 2 years quarterly interest paying fd closed on 1 jan 2025. howmuch will be settlement?
🔍 Running validation check...
🤖 Running Agent-1 (Extraction)...
PRIMARY RESP:
{
  "principal": 120000,
  "start_date": "2023-01-01",
  "tenure_value": 2,
  "tenure_unit": "years",
  "withdrawal_date": "2025-01-01",
  "holding_days": null,
  "payout_type": "quarterly_payout",
  "is_premature": true,
  "citizenship_status": "regular",
  "nri_status": "Non NRI",
  "deposit_amount": "<3CR"
}
Tenure engine: 2 years from 2023-01-01 → maturity 2025-01-01 (731 days)
Correcting holding_days: None → 731
✅ Validating Agent-1 output...
🤖 Running Agent-2 (Planning)...
💰 Running Agent-4 (Premature Withdrawal)...

RESULT:
{
  "route": "calculation_premature",
  "final_answer": "Query: \"FD booked of amt 120,000 on 1 Jan 2023 for 2 years quarterly interest paying fd closed on 1 Jan 2025. how much will be settlement?\"\n\n## Premature Withdrawal Calculation\n\n### Step